# Pokemon TCG AI Battle Challenge - Lucario v2 Strategic Baseline

Pokemon TCG rewards stable legal-action ranking under hidden hands, hidden prizes, random draws, and matchup-specific threats.

This Lucario v2 agent keeps Mega Lucario ex as the main pressure line, but it now plays as a meta-aware rule policy: safe setup first, selective knockouts second, and no optional action unless it has a positive tactical reason.

In [1]:
from pathlib import Path
import glob
import json
import math
import os
import random
import sys
import shutil
import tarfile
import textwrap

try:
    import numpy as np
except Exception:
    np = None

try:
    import pandas as pd
except Exception:
    pd = None

try:
    from IPython.display import display
except Exception:
    display = print

SEED = 20260617
N_SMOKE_GAMES = 60
N_MATCHUP_GAMES = 180
LOW_DECK_COUNT = 8
USE_SEARCH = False
WRITE_SUBMISSION = True
REQUIRE_CG_IN_ARCHIVE = True
ENFORCE_GATE = True            # if True, a failed validation gate raises and blocks submission
WINRATE_GATE = 0.80            # min win-rate vs legal-random (alternating seat)
FALLBACK_GATE = 0.005          # max fraction of decisions allowed to hit the safety fallback
SEARCH_FAIL_GATE = 0.0         # search must be clean before USE_SEARCH is enabled
MAX_STEPS_PER_GAME = 2000      # hard cap so a stuck game cannot hang validation
USE_SEARCH_IN_AGENT = False    # OFF by default; enable only after search A/B passes the failure and win-rate gates
SEARCH_PROFILE = "live"         # "live" keeps current 800+ settings; "safe" lowers search cost
SEARCH_NODE_BUDGET = 120       # max engine forward-steps per decision
SEARCH_TIME_BUDGET_S = 1.5     # wall-clock cap per decision (respect the 10-min/game limit)
SEARCH_ACTION_CAP = 20         # max root MAIN action combinations evaluated by search
if SEARCH_PROFILE == "safe":
    SEARCH_NODE_BUDGET = min(SEARCH_NODE_BUDGET, 80)
    SEARCH_TIME_BUDGET_S = min(SEARCH_TIME_BUDGET_S, 1.0)
    SEARCH_ACTION_CAP = min(SEARCH_ACTION_CAP, 12)
DECK_VARIANT = "public1084_energy_hero"  # options: baseline, tempo, keep_hero_tempo, energy_consistency, public1084_energy_hero, crustle_retuned
CRUSTLE_AWARE_IN_AGENT = True

ROOT = Path.cwd()
SAMPLE_DECK_DIR = ROOT / "Sample_Decks"
PUBLIC_EXAMPLES_DIR = ROOT / "Public_Examples"
BUILD_DIR = ROOT / "phase1_lucario_v2_build"
SUBMISSION_PATH = ROOT / "submission.tar.gz"

random.seed(SEED)
if np is not None:
    np.random.seed(SEED)

print("ROOT:", ROOT)
print("Sample decks:", SAMPLE_DECK_DIR.exists(), SAMPLE_DECK_DIR)
print("Public examples:", PUBLIC_EXAMPLES_DIR.exists(), PUBLIC_EXAMPLES_DIR)
print("Require cg/ in final archive:", REQUIRE_CG_IN_ARCHIVE)



ROOT: /kaggle/working
Sample decks: False /kaggle/working/Sample_Decks
Public examples: False /kaggle/working/Public_Examples
Require cg/ in final archive: True


## Deck Engine

The active list is the high-score keep-Hero branch: extra Riolu, Boss, and Energy while preserving Hero Cape.

Core roles:

- **Mega Lucario ex**: primary tempo and closing attacker.
- **Hariyama / Makuhita**: non-ex wall breaker, especially into Crustle.
- **Solrock / Lunatone**: light backup line and setup support.
- **Boss / Switch / Gravity Mountain**: convert attack plans into knockouts.
- **Dusk Ball / Poke Pad / Carmine / Lillie**: setup consistency, gated by deck-out and win-now safety.

In [2]:
BASELINE_DECK = [
    673, 673, 674, 674, 675, 675, 676, 676,
    676, 677, 677, 677, 678, 678, 678, 678,
    1102, 1102, 1102, 1102, 1123, 1123, 1141, 1141,
    1141, 1141, 1142, 1142, 1142, 1142, 1152, 1152,
    1152, 1152, 1159, 1182, 1182, 1192, 1192, 1192,
    1192, 1227, 1227, 1227, 1227, 1252, 1252, 6,
    6, 6, 6, 6, 6, 6, 6, 6,
    6, 6, 6, 6,
]

CARD_LABELS = {
    673: ("Makuhita", "Pokemon", "Basic"),
    674: ("Hariyama", "Pokemon", "Stage 1"),
    675: ("Lunatone", "Pokemon", "Basic"),
    676: ("Solrock", "Pokemon", "Basic"),
    677: ("Riolu", "Pokemon", "Basic"),
    678: ("Mega Lucario ex", "Pokemon", "Mega ex"),
    1102: ("Dusk Ball", "Trainer", "Item"),
    1123: ("Switch", "Trainer", "Item"),
    1141: ("Premium Power Pro", "Trainer", "Tool"),
    1142: ("Fighting Gong", "Trainer", "Item"),
    1152: ("Poke Pad", "Trainer", "Item"),
    1159: ("Hero Cape", "Trainer", "Tool"),
    1182: ("Boss's Orders", "Trainer", "Supporter"),
    1192: ("Carmine", "Trainer", "Supporter"),
    1227: ("Lillie's Determination", "Trainer", "Supporter"),
    1252: ("Gravity Mountain", "Trainer", "Stadium"),
    6: ("Basic Fighting Energy", "Energy", "Basic Energy"),
}

from collections import Counter

assert len(BASELINE_DECK) == 60
assert all(isinstance(x, int) for x in BASELINE_DECK)
assert set(BASELINE_DECK) <= set(CARD_LABELS), "Every deck card should have a notebook label."

# Deck variants use only cards already understood by the policy.
# tempo: +Boss/+Riolu, -Gravity/-HeroCape for mirror pressure.
# keep_hero_tempo: keeps the defensive Hero Cape out while trimming Poke Pad instead.
# energy_consistency: adds a Basic Energy and Riolu, trading off a search card and spare Stadium.
# public1084_energy_hero: high-score conservative list, restoring Hero Cape and adding one Energy.
# crustle_retuned: stronger Makuhita/Hariyama/Switch density for the Crustle wall.

def _apply(deck, card_id, delta):
    d = list(deck)
    if delta < 0:
        for _ in range(-delta):
            d.remove(card_id)
    else:
        d.extend([card_id] * delta)
    return d

TEMPO_DECK = list(BASELINE_DECK)
TEMPO_DECK = _apply(TEMPO_DECK, 1182, +1)
TEMPO_DECK = _apply(TEMPO_DECK, 677, +1)
TEMPO_DECK = _apply(TEMPO_DECK, 1252, -1)
TEMPO_DECK = _apply(TEMPO_DECK, 1159, -1)

KEEP_HERO_TEMPO_DECK = list(BASELINE_DECK)
KEEP_HERO_TEMPO_DECK = _apply(KEEP_HERO_TEMPO_DECK, 1182, +1)
KEEP_HERO_TEMPO_DECK = _apply(KEEP_HERO_TEMPO_DECK, 677, +1)
KEEP_HERO_TEMPO_DECK = _apply(KEEP_HERO_TEMPO_DECK, 1252, -1)
KEEP_HERO_TEMPO_DECK = _apply(KEEP_HERO_TEMPO_DECK, 1152, -1)

ENERGY_CONSISTENCY_DECK = list(BASELINE_DECK)
ENERGY_CONSISTENCY_DECK = _apply(ENERGY_CONSISTENCY_DECK, 677, +1)
ENERGY_CONSISTENCY_DECK = _apply(ENERGY_CONSISTENCY_DECK, 6, +1)
ENERGY_CONSISTENCY_DECK = _apply(ENERGY_CONSISTENCY_DECK, 1252, -1)
ENERGY_CONSISTENCY_DECK = _apply(ENERGY_CONSISTENCY_DECK, 1152, -1)

PUBLIC1084_ENERGY_HERO_DECK = list(BASELINE_DECK)
PUBLIC1084_ENERGY_HERO_DECK = _apply(PUBLIC1084_ENERGY_HERO_DECK, 677, +1)
PUBLIC1084_ENERGY_HERO_DECK = _apply(PUBLIC1084_ENERGY_HERO_DECK, 1182, +1)
PUBLIC1084_ENERGY_HERO_DECK = _apply(PUBLIC1084_ENERGY_HERO_DECK, 6, +1)
PUBLIC1084_ENERGY_HERO_DECK = _apply(PUBLIC1084_ENERGY_HERO_DECK, 1252, -1)
PUBLIC1084_ENERGY_HERO_DECK = _apply(PUBLIC1084_ENERGY_HERO_DECK, 1152, -2)

CRUSTLE_RETUNED_DECK = (
    [678] * 4 + [677] * 4 + [673] * 3 + [674] * 3 + [676] * 3 + [675] * 2
    + [1102] * 4 + [1152] * 4 + [1192] * 4 + [1142] * 3 + [1123] * 3
    + [1141] * 2 + [1227] * 3 + [1252] * 2 + [1182] * 2 + [1159] * 1 + [6] * 13
)

DECK_VARIANTS = {
    "baseline": BASELINE_DECK,
    "tempo": TEMPO_DECK,
    "keep_hero_tempo": KEEP_HERO_TEMPO_DECK,
    "energy_consistency": ENERGY_CONSISTENCY_DECK,
    "public1084_energy_hero": PUBLIC1084_ENERGY_HERO_DECK,
    "crustle_retuned": CRUSTLE_RETUNED_DECK,
}
assert DECK_VARIANT in DECK_VARIANTS, sorted(DECK_VARIANTS)
DECK = DECK_VARIANTS[DECK_VARIANT]
counts = Counter(DECK)
basic_ids = {673, 675, 676, 677}
basic_count = sum(counts[i] for i in basic_ids)
prob_no_basic_opening_7 = math.comb(len(DECK) - basic_count, 7) / math.comb(len(DECK), 7)

assert len(DECK) == 60, len(DECK)
assert set(DECK) <= set(CARD_LABELS), "Every active-deck card should have a notebook label."
non_energy_over_4 = [card_id for card_id, count in counts.items() if card_id != 6 and count > 4]
assert not non_energy_over_4, non_energy_over_4
assert basic_count >= 1, "active deck has no Basic Pokemon"

rows = []
for card_id, count in sorted(counts.items(), key=lambda kv: (CARD_LABELS[kv[0]][1], CARD_LABELS[kv[0]][0])):
    name, card_type, subtype = CARD_LABELS[card_id]
    rows.append({"card_id": card_id, "name": name, "count": count, "type": card_type, "subtype": subtype})

if pd is not None:
    deck_df = pd.DataFrame(rows)
    display(deck_df)
    display(deck_df.groupby("type", as_index=False)["count"].sum())
else:
    for row in rows:
        print(row)

BUILD_DIR.mkdir(exist_ok=True)
(BUILD_DIR / "deck.csv").write_text("\n".join(map(str, DECK)) + "\n")

print(f"Active deck variant: {DECK_VARIANT}")
print("deck length:", len(DECK))
print("unique cards:", len(counts))
print("basic pokemon count:", basic_count)
print("P(no Basic in opening 7):", round(prob_no_basic_opening_7, 4))
print("deck.csv:", BUILD_DIR / "deck.csv")

,card_id,name,count,type,subtype
0,6,Basic Fighting Energy,14,Energy,Basic Energy
1,674,Hariyama,2,Pokemon,Stage 1
2,675,Lunatone,2,Pokemon,Basic
3,673,Makuhita,2,Pokemon,Basic
4,678,Mega Lucario ex,4,Pokemon,Mega ex
5,677,Riolu,4,Pokemon,Basic
6,676,Solrock,3,Pokemon,Basic
7,1182,Boss's Orders,3,Trainer,Supporter
8,1192,Carmine,4,Trainer,Supporter
9,1102,Dusk Ball,4,Trainer,Item


,type,count
0,Energy,14
1,Pokemon,17
2,Trainer,29


Active deck variant: public1084_energy_hero
deck length: 60
unique cards: 17
basic pokemon count: 11
P(no Basic in opening 7): 0.2224
deck.csv: /kaggle/working/phase1_lucario_v2_build/deck.csv


## Decision Stack

Each turn is handled in one pass:

1. Read the legal `select.option` list.
2. Build a turn-local `AttackPlan`.
3. Score targets by prizes, setup value, energy, tools, stage, HP, and visible archetype.
4. Score legal options by context.
5. Normalize the final selection with `score > 0` for optional picks and `minCount` for mandatory picks.

The policy remains greedy by default; search is a gated experiment, not the live path.

In [3]:
MAIN_PRELUDE = r'''from __future__ import annotations

import os
from collections import defaultdict
import time
import random

try:
    from cg.api import search_begin, search_step, search_release
    _SEARCH_AVAILABLE = True
except Exception:
    try:
        from cg.api import search_begin, search_step, search_end as search_release
        _SEARCH_AVAILABLE = True
    except Exception:
        _SEARCH_AVAILABLE = False

from cg.api import (
    AreaType,
    Card,
    CardType,
    EnergyType,
    Observation,
    OptionType,
    Pokemon,
    SelectContext,
    all_card_data,
    to_observation_class,
)


class C:
    DWEBBLE = 344
    CRUSTLE = 345

    KYOGRE = 721
    SNOVER = 722
    MEGA_ABOMASNOW_EX = 723

    MAKUHITA = 673
    HARIYAMA = 674
    LUNATONE = 675
    SOLROCK = 676
    RIOLU = 677
    MEGA_LUCARIO_EX = 678

    BASIC_FIGHTING_ENERGY = 6
    DUSK_BALL = 1102
    SWITCH = 1123
    PREMIUM_POWER_PRO = 1141
    FIGHTING_GONG = 1142
    POKE_PAD = 1152
    HERO_CAPE = 1159
    BOSS_ORDERS = 1182
    CARMINE = 1192
    LILLIE_DETERMINATION = 1227
    GRAVITY_MOUNTAIN = 1252

    LUMIOSE_CITY = 1267
    LILLIES_PEARL = 1172
    LEGACY_ENERGY = 12


MEGA_BRAVE = 983
LOW_DECK_COUNT = 8
CRUSTLE_AWARE = True

EMBEDDED_DECK = [
    673, 673, 674, 674, 675, 675, 676, 676,
    676, 677, 677, 677, 678, 678, 678, 678,
    1102, 1102, 1102, 1102, 1123, 1123, 1141, 1141,
    1141, 1141, 1142, 1142, 1142, 1142, 1152, 1152,
    1152, 1152, 1159, 1182, 1182, 1192, 1192, 1192,
    1192, 1227, 1227, 1227, 1227, 1252, 1252, 6,
    6, 6, 6, 6, 6, 6, 6, 6,
    6, 6, 6, 6,
]


def _resolve_deck_path() -> str:
    base_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    candidates = [
        os.path.join(base_dir, "deck.csv"),
        "deck.csv",
        "/kaggle_simulations/agent/deck.csv",
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError("deck.csv not found in: " + ", ".join(candidates))


def _load_deck() -> list[int]:
    try:
        deck_path = _resolve_deck_path()
        with open(deck_path, "r", encoding="utf-8") as f:
            deck = [int(line) for line in f.read().splitlines() if line.strip()]
        if len(deck) == 60:
            return deck
    except Exception:
        pass
    return EMBEDDED_DECK.copy()


my_deck = _load_deck()
if len(my_deck) != 60:
    raise ValueError(f"deck must contain 60 card ids, got {len(my_deck)}")


all_card = all_card_data()
card_table = {card.cardId: card for card in all_card}


class AttackPlan:
    def __init__(
        self,
        attacker: int = -1,
        target: int = -1,
        attack_index: int = -1,
        remain_hp: int = -1,
        needs_energy: bool = False,
    ):
        self.attacker = attacker
        self.target = target
        self.attack_index = attack_index
        self.remain_hp = remain_hp
        self.needs_energy = needs_energy


plan = AttackPlan()
pre_turn = -1
ability_used = False


def _has_attack_plan() -> bool:
    return plan.attacker >= 0 and plan.target >= 0 and plan.attack_index >= 0


def _plan_kos() -> bool:
    return _has_attack_plan() and plan.remain_hp <= 0

# Optional search hook. The Phase 1 notebook keeps this disabled; it exists so later
# experiments can be connected without changing the submission contract.
USE_SEARCH = False
'''

In [4]:
MAIN_DIAGNOSTICS = r'''# --- Phase-1 diagnostics: distinguish "policy ran" from "fell back". ---------
# A submission that never crashes can still be silently broken: if the policy
# touches an SDK attribute that does not exist on the live engine, the exception
# is caught and we return _legal_fallback (the first minCount indices). "0 errors"
# then hides a near-random agent. _DIAG makes the fallback rate and the actual
# exception messages observable so the validation gate can catch this.
_DIAG = {"decisions": 0, "policy_ok": 0, "policy_fallback": 0,
         "obs_fallback": 0, "deck_returns": 0, "errors": {},
         "policy_errors": {}, "search_errors": {},
         "search_used": 0, "search_failed": 0, "search_disabled": 0,
         "chosen_types": {}, "attack_ids_chosen": {},
         "attack_opts_by_active": {}, "mega_brave_present": 0,
         "yes_no_contexts": {}, "crustle_seen": 0,
         "crustle_active_seen": 0, "crustle_wall_policy_seen": 0,
         "crustle_ex_attack_options": 0,
         "crustle_ex_attack_suppressed": 0,
         "crustle_ex_attack_chosen": 0,
         "crustle_non_ex_attack_chosen": 0,
         "crustle_final_plan_ex_into_wall": 0,
         "crustle_final_plan_non_ex_into_wall": 0}


def _diag_record_error(exc):
    key = type(exc).__name__ + ": " + str(exc)[:160]
    _DIAG["errors"][key] = _DIAG["errors"].get(key, 0) + 1
    _DIAG["policy_errors"][key] = _DIAG["policy_errors"].get(key, 0) + 1


def _diag_record_search_error(exc):
    key = type(exc).__name__ + ": " + str(exc)[:160]
    _DIAG["errors"][key] = _DIAG["errors"].get(key, 0) + 1
    _DIAG["search_errors"][key] = _DIAG["search_errors"].get(key, 0) + 1


def diag_reset():
    _DIAG.update({"decisions": 0, "policy_ok": 0, "policy_fallback": 0,
                  "obs_fallback": 0, "deck_returns": 0, "errors": {},
                  "policy_errors": {}, "search_errors": {},
                  "search_used": 0, "search_failed": 0, "search_disabled": 0,
                  "chosen_types": {}, "attack_ids_chosen": {},
                  "attack_opts_by_active": {}, "mega_brave_present": 0,
                  "yes_no_contexts": {}, "crustle_seen": 0,
                  "crustle_active_seen": 0, "crustle_wall_policy_seen": 0,
                  "crustle_ex_attack_options": 0,
                  "crustle_ex_attack_suppressed": 0,
                  "crustle_ex_attack_chosen": 0,
                  "crustle_non_ex_attack_chosen": 0,
                  "crustle_final_plan_ex_into_wall": 0,
                  "crustle_final_plan_non_ex_into_wall": 0})


def diag_snapshot():
    snap = {}
    for k, v in _DIAG.items():
        if k == "attack_opts_by_active":
            snap[k] = {kk: sorted(vv) for kk, vv in v.items()}
        elif isinstance(v, dict):
            snap[k] = dict(v)
        else:
            snap[k] = v
    dec = max(1, snap.get("decisions", 0))
    snap["fallback_rate"] = (snap.get("policy_fallback", 0) + snap.get("obs_fallback", 0)) / dec
    sd = snap.get("search_used", 0) + snap.get("search_failed", 0)
    snap["search_fail_rate"] = (snap.get("search_failed", 0) / sd) if sd else 0.0
    return snap


def _diag_observe(obs):
    """Record, on the REAL obs, which attacks are legal (reveals true attackIds)."""
    try:
        sel = obs.select
        st = obs.current
        if sel is None or st is None:
            return
        me = st.players[st.yourIndex]
        op = st.players[1 - st.yourIndex]
        active_id = me.active[0].id if (me.active and me.active[0] is not None) else -1
        op_active_id = op.active[0].id if (op.active and op.active[0] is not None) else -1
        opponent_ids = [p.id for p in (op.active + op.bench) if p is not None]
        crustle_seen = any(cid in {C.DWEBBLE, C.CRUSTLE} for cid in opponent_ids)
        if crustle_seen:
            _DIAG["crustle_seen"] += 1
        if op_active_id == C.CRUSTLE:
            _DIAG["crustle_active_seen"] += 1
        for opt in sel.option:
            if opt.type == OptionType.ATTACK:
                aid = getattr(opt, "attackId", None)
                if aid is not None:
                    _DIAG["attack_opts_by_active"].setdefault(active_id, set()).add(aid)
                    if active_id == C.MEGA_LUCARIO_EX and op_active_id == C.CRUSTLE:
                        _DIAG["crustle_ex_attack_options"] += 1
                    if aid == MEGA_BRAVE:
                        _DIAG["mega_brave_present"] += 1
            if opt.type in {OptionType.YES, OptionType.NO}:
                ctx = str(getattr(sel, "context", None))
                _DIAG["yes_no_contexts"][ctx] = _DIAG["yes_no_contexts"].get(ctx, 0) + 1
    except Exception:
        pass


def _diag_observe_choice(obs, selection):
    """Record which option TYPES (and attackIds) actually get chosen."""
    try:
        sel = obs.select
        st = obs.current
        if sel is None or st is None or not selection:
            return
        me = st.players[st.yourIndex]
        op = st.players[1 - st.yourIndex]
        active_id = me.active[0].id if (me.active and me.active[0] is not None) else -1
        op_active_id = op.active[0].id if (op.active and op.active[0] is not None) else -1
        for i in selection:
            if 0 <= i < len(sel.option):
                opt = sel.option[i]
                key = str(opt.type)
                _DIAG["chosen_types"][key] = _DIAG["chosen_types"].get(key, 0) + 1
                if opt.type == OptionType.ATTACK:
                    aid = getattr(opt, "attackId", None)
                    if aid is not None:
                        _DIAG["attack_ids_chosen"][aid] = _DIAG["attack_ids_chosen"].get(aid, 0) + 1
                    if op_active_id == C.CRUSTLE:
                        if active_id == C.MEGA_LUCARIO_EX:
                            _DIAG["crustle_ex_attack_chosen"] += 1
                        elif active_id in {C.HARIYAMA, C.MAKUHITA, C.SOLROCK}:
                            _DIAG["crustle_non_ex_attack_chosen"] += 1
    except Exception:
        pass
'''

## Optional Search Layer

Forward search is kept as a guarded experiment. It only runs when the SDK exposes compatible search input, the position is high leverage, and previous search failures are below the gate.

The live profile keeps `USE_SEARCH = False`. The stable path is the rule policy plus diagnostics.

In [5]:
MAIN_FORWARD_SEARCH = r'''# ---------------------------------------------------------------------------
# Forward-search over the engine model. This uses the live SDK convention:
# the observation dictionary carries search_begin_input, search_begin returns a
# wrapper with {error, state}, search_step advances a searchId, and search_release
# must be called for every started branch.
# ---------------------------------------------------------------------------
SEARCH_NODE_BUDGET = 120        # max forward-steps per decision
SEARCH_TIME_BUDGET_S = 1.5      # wall-clock cap per decision (respect 10-min/game)
SEARCH_ACTION_CAP = 20          # root MAIN candidates evaluated by search
SEARCH_FAILURE_DISABLE_AFTER = 8
SEARCH_MIN_ATTEMPTS_FOR_RATE = 10
SEARCH_FAIL_RATE_DISABLE = 0.75
SEARCH_OPPONENT_PLY = False     # opponent reply search is experimental, off by default


def _search_temporarily_disabled():
    attempts = _DIAG.get("search_used", 0) + _DIAG.get("search_failed", 0)
    failed = _DIAG.get("search_failed", 0)
    used = _DIAG.get("search_used", 0)
    if failed >= SEARCH_FAILURE_DISABLE_AFTER and used == 0:
        return True
    if attempts >= SEARCH_MIN_ATTEMPTS_FOR_RATE and failed / max(1, attempts) >= SEARCH_FAIL_RATE_DISABLE:
        return True
    return False


def _board_value(obs, my_index):
    """Win-aligned leaf score used only by optional forward search."""
    st = obs.current
    res = getattr(st, "result", -1)
    if res is not None and res >= 0:
        if res == my_index:
            return 1_000_000.0
        if res == (1 - my_index):
            return -1_000_000.0
        return 0.0

    me = st.players[my_index]
    op = st.players[1 - my_index]
    val = 10_000.0 * (len(op.prize) - len(me.prize))

    my_active = me.active[0] if me.active else None
    op_active = op.active[0] if op.active else None
    if op_active is not None:
        val += 2.0 * _damage_on(op_active)
        val += 800.0 * prize_count(op_active)
    if my_active is not None:
        val -= 1.5 * _damage_on(my_active)
        val -= 600.0 * prize_count(my_active)

    ready_attackers = 0
    for pkm in (me.active + me.bench):
        if pkm is None:
            continue
        energy_count = len(getattr(pkm, "energies", []))
        if pkm.id == C.MEGA_LUCARIO_EX and energy_count >= 2:
            ready_attackers += 1
        elif pkm.id == C.HARIYAMA and energy_count >= 3:
            ready_attackers += 1
        elif pkm.id == C.SOLROCK and energy_count >= 1:
            ready_attackers += 1
        val += 30.0 * energy_count

    val += 300.0 * ready_attackers
    val += 20.0 * getattr(me, "handCount", 0)
    if getattr(me, "deckCount", 0) <= 4:
        val -= 500.0 * (5 - getattr(me, "deckCount", 0))
    return val


def _should_search(obs, ranked, scores):
    sel = obs.select
    if sel is None or obs.current is None:
        return False
    if sel.context != SelectContext.MAIN:
        return False
    if len(sel.option) <= 1:
        return False
    option_types = {opt.type for opt in sel.option}
    tactical_types = {
        OptionType.ATTACK,
        OptionType.ATTACH,
        OptionType.EVOLVE,
        OptionType.RETREAT,
        OptionType.PLAY,
    }
    if not (option_types & tactical_types):
        return False
    if _plan_kos():
        return True
    if plan.target >= 1:
        return True
    if plan.attacker > 0:
        return True
    if plan.needs_energy:
        return True
    return False


def _result_state(result):
    if result is None:
        return None
    if isinstance(result, dict):
        error = result.get("error", 0)
        state = result.get("state")
    else:
        error = getattr(result, "error", 0)
        state = getattr(result, "state", None)
    if error not in (0, None):
        return None
    if state is not None:
        return state
    if hasattr(result, "observation") or hasattr(result, "searchId"):
        return result
    return None


def _state_id(state):
    if isinstance(state, dict):
        return state.get("searchId")
    return getattr(state, "searchId", None)


def _state_obs(state):
    if isinstance(state, dict):
        return state.get("observation")
    return getattr(state, "observation", None)


def _release_search(sid):
    try:
        search_release(sid)
    except TypeError:
        try:
            search_release()
        except Exception:
            pass
    except Exception:
        pass


def _search_step_state(sid, selection):
    if sid is None:
        return None
    return _result_state(search_step(sid, selection))


def _legal_sim_selection(obs, preferred_first=None):
    try:
        policy = LucarioPolicy(obs)
        ranked, scores = policy.rank()
        if preferred_first is not None:
            ranked = [preferred_first] + [i for i in ranked if i != preferred_first]
        selection = normalize_selection(ranked, scores, obs.select)
        return selection if selection else _legal_fallback(obs.select)
    except Exception:
        return _legal_fallback(obs.select)


def _rollout_search_candidate(sbi, first_idx, my_index, t0):
    sid = None
    steps = 0
    cur = None
    try:
        state = _result_state(search_begin(sbi))
        if state is None:
            return None, steps
        sid = _state_id(state)
        cur = _state_obs(state)
        if cur is None or cur.select is None:
            return None, steps

        selection = _legal_sim_selection(cur, preferred_first=first_idx)
        while steps < SEARCH_NODE_BUDGET and (time.time() - t0) <= SEARCH_TIME_BUDGET_S:
            state = _search_step_state(sid, selection)
            if state is None:
                return None, steps
            steps += 1
            cur = _state_obs(state)
            if cur is None or cur.current is None:
                break
            result = getattr(cur.current, "result", -1)
            if result is not None and result >= 0:
                break
            if cur.current.yourIndex != my_index:
                break
            if cur.select is None or len(cur.select.option) == 0:
                break
            selection = _legal_sim_selection(cur)
            if not selection:
                break
            if cur.select.context == SelectContext.MAIN:
                try:
                    if any(cur.select.option[i].type == OptionType.END for i in selection if 0 <= i < len(cur.select.option)):
                        state = _search_step_state(sid, selection)
                        if state is not None:
                            cur = _state_obs(state) or cur
                            steps += 1
                        break
                except Exception:
                    pass
        if cur is None:
            return None, steps
        return _board_value(cur, my_index), steps
    finally:
        if sid is not None:
            _release_search(sid)


def search_plan(obs_dict, obs, ranked, scores):
    if not _SEARCH_AVAILABLE:
        return None
    if _search_temporarily_disabled():
        _DIAG["search_disabled"] += 1
        return None
    try:
        if obs.select is None or obs.current is None:
            return None
        if obs.select.context != SelectContext.MAIN:
            return None
        if obs.select.maxCount == 0 or len(obs.select.option) <= 1:
            return None
        if not _should_search(obs, ranked, scores):
            return None
        sbi = getattr(obs, "search_begin_input", None)
        if sbi is None and isinstance(obs_dict, dict):
            sbi = obs_dict.get("search_begin_input")
        if sbi is None:
            return None

        my_index = obs.current.yourIndex
        candidates = []
        for idx in ranked:
            if not (0 <= idx < len(obs.select.option)) or idx in candidates:
                continue
            score = scores[idx] if idx < len(scores) else 0
            if score <= 0 and obs.select.minCount == 0:
                continue
            candidates.append(idx)
            if len(candidates) >= SEARCH_ACTION_CAP:
                break
        if not candidates:
            return None

        t0 = time.time()
        best_idx, best_val = None, None
        total_steps = 0
        for first_idx in candidates:
            if total_steps >= SEARCH_NODE_BUDGET or (time.time() - t0) > SEARCH_TIME_BUDGET_S:
                break
            value, used_steps = _rollout_search_candidate(sbi, first_idx, my_index, t0)
            total_steps += max(1, used_steps)
            if value is None:
                continue
            if best_val is None or value > best_val:
                best_idx, best_val = first_idx, value
        if best_idx is None:
            _DIAG["search_failed"] += 1
            return None
        _DIAG["search_used"] += 1
        return [best_idx] + [i for i in ranked if i != best_idx]
    except Exception as exc:
        _diag_record_search_error(exc)
        _DIAG["search_failed"] += 1
        return None
'''

## Context-Aware Score Floor

Pokemon TCG choices share one index interface, but the meanings differ: setup, attach, evolve, discard, damage counters, retreat, attack, and card search all need different scoring.

The selection filter therefore treats zero as unknown, not useful. Optional choices need `score > 0`; mandatory contexts still satisfy `minCount`. This prevents the agent from filling `maxCount` with harmless-looking but harmful extras.

In [6]:
MAIN_SELECTION_HELPERS = r'''def normalize_selection(ranked, scores, select):
    n = len(select.option)
    minc = max(0, min(select.minCount, n))
    maxc = max(minc, min(select.maxCount, n))

    out = []
    seen = set()
    for i in ranked:
        if not (0 <= i < n) or i in seen:
            continue
        score = scores[i] if i < len(scores) else 0
        # Score 0 usually means an unmodelled neutral option. For optional
        # selections, do not take it unless minCount forces a choice.
        if score > 0 or len(out) < minc:
            out.append(i)
            seen.add(i)
        if len(out) >= maxc:
            break

    for i in range(n):
        if len(out) >= minc:
            break
        if i not in seen:
            out.append(i)
            seen.add(i)
    return out


def _legal_fallback(select):
    try:
        n = len(select.option)
        k = min(max(0, select.minCount), n)
        return list(range(k))
    except Exception:
        return []


def _legal_fallback_from_dict(obs_dict):
    try:
        sel = obs_dict.get("select") or {}
        opts = sel.get("option") or []
        minc = sel.get("minCount", 0)
        n = len(opts)
        k = min(max(0, minc), n)
        return list(range(k))
    except Exception:
        return []

def _safe_get(seq, index: int):
    try:
        if seq is None or index is None or index < 0 or index >= len(seq):
            return None
        return seq[index]
    except Exception:
        return None
'''

In [7]:
MAIN_CARD_HELPERS = r'''def _ctx_is(context, *names) -> bool:
    for name in names:
        value = getattr(SelectContext, name, None)
        if value is not None and context == value:
            return True
    return False


def _pokemon_max_hp(pokemon: Pokemon) -> int:
    max_hp = getattr(pokemon, "maxHp", None)
    if max_hp is not None:
        return max_hp
    data = card_table.get(getattr(pokemon, "id", None))
    return getattr(data, "hp", getattr(pokemon, "hp", 0))


def _damage_on(pokemon: Pokemon) -> int:
    return max(0, _pokemon_max_hp(pokemon) - getattr(pokemon, "hp", 0))


def get_card(obs: Observation, area: AreaType, index: int, player_index: int) -> Pokemon | Card | None:
    try:
        player = obs.current.players[player_index]
        match area:
            case AreaType.DECK:
                return _safe_get(getattr(obs.select, "deck", None), index)
            case AreaType.HAND:
                return _safe_get(getattr(player, "hand", None), index)
            case AreaType.DISCARD:
                return _safe_get(getattr(player, "discard", None), index)
            case AreaType.ACTIVE:
                return _safe_get(getattr(player, "active", None), index)
            case AreaType.BENCH:
                return _safe_get(getattr(player, "bench", None), index)
            case AreaType.PRIZE:
                return _safe_get(getattr(player, "prize", None), index)
            case AreaType.STADIUM:
                return _safe_get(getattr(obs.current, "stadium", None), index)
            case AreaType.LOOKING:
                return _safe_get(getattr(obs.current, "looking", None), index)
            case _:
                return None
    except Exception:
        return None


def prize_count(pokemon: Pokemon) -> int:
    data = card_table.get(pokemon.id)
    if data is None:
        return 1
    count = 3 if data.megaEx else 2 if data.ex else 1
    for card in pokemon.energyCards:
        if card.id == C.LEGACY_ENERGY:
            count -= 1
    for card in pokemon.tools:
        if card.id == C.LILLIES_PEARL and "Lillie" in data.name:
            count -= 1
    return max(0, count)


def target_score(pokemon: Pokemon) -> int:
    data = card_table.get(pokemon.id)
    if data is None:
        return prize_count(pokemon) * 1000 + getattr(pokemon, "hp", 0) + _damage_on(pokemon) * 2
    score = prize_count(pokemon) * 1000
    score += len(pokemon.energies) * 150
    score += len(pokemon.tools) * 100
    if data.stage2:
        score += 250
    elif data.stage1:
        score += 130

    if pokemon.id in {144, 322, 323, 337}:  # low-value support Pokemon
        score -= 200
    if pokemon.id == C.SNOVER:
        score += 950
    elif pokemon.id == C.MEGA_ABOMASNOW_EX:
        score += 250
    if pokemon.id == C.RIOLU:
        score += 800
    elif pokemon.id == C.MEGA_LUCARIO_EX:
        score += 100
    if pokemon.id == 112 and len(pokemon.energies) >= 1:  # Munkidori
        score += 300
    score += pokemon.hp
    return score
'''

## Lucario Policy Body

The policy is split by function: card access, target scoring, attack planning, option scoring, trainer handling, attachment, evolution, and diagnostics side effects.

Current strategic edges:

- Crustle wall detection blocks ex attacks into damage immunity.
- Hariyama energy, evolution, active choice, and retreat become stronger when Crustle is visible.
- Water/Abomasnow visibility lowers late ex exposure and raises Hero Cape priority.
- Target scoring prioritizes visible setup pieces such as Riolu and Snover.
- Win-now suppression avoids extra draw/search actions before a known knockout.
- Forward search remains disabled until the engine API path is proven stable.

In [8]:
MAIN_POLICY = r'''class LucarioPolicy:
    def __init__(self, obs: Observation):
        self.obs = obs
        self.state = obs.current
        self.select = obs.select
        self.context = self.select.context
        self.my_index = self.state.yourIndex
        self.op_index = 1 - self.my_index
        self.me = self.state.players[self.my_index]
        self.opponent = self.state.players[self.op_index]
        self.my_prizes_left = len(self.me.prize)

        self.field_counts = defaultdict(int)
        self.hand_counts = defaultdict(int)
        self.discard_counts = defaultdict(int)
        self.has_ready_lucario_line = False
        self.has_ready_hariyama_line = False
        self.can_switch = False
        self.can_gust = False
        self.can_attack = False
        self.can_use_mega_brave = False
        self.stadium_id = self.state.stadium[0].id if self.state.stadium else 0
        self.crustle_wall = self._opponent_is_crustle_wall()
        self.water_deck = self._opponent_is_water_deck()
        if self.crustle_wall:
            _DIAG["crustle_wall_policy_seen"] += 1

        self._count_cards()
        self._scan_main_options()

    def rank(self) -> tuple[list[int], list[float]]:
        if not self.select.option or self.select.maxCount == 0:
            return [], []

        if self.context == SelectContext.MAIN:
            self._plan_attack()

        scores = [self._score_option(option) for option in self.select.option]
        ranked = [i for i, _ in sorted(enumerate(scores), key=lambda item: item[1], reverse=True)]
        return ranked, scores

    def choose(self) -> list[int]:
        ranked, scores = self.rank()
        selection = normalize_selection(ranked, scores, self.select)
        self.remember_chosen_side_effects(selection)
        return selection
    def _count_cards(self) -> None:
        for pokemon in self.me.active + self.me.bench:
            if pokemon is None:
                continue
            self.field_counts[pokemon.id] += 1
            if pokemon.id in {C.MAKUHITA, C.HARIYAMA} and len(pokemon.energies) >= 3:
                self.has_ready_hariyama_line = True
            if pokemon.id in {C.RIOLU, C.MEGA_LUCARIO_EX} and len(pokemon.energies) >= 2:
                self.has_ready_lucario_line = True

        for card in self.me.hand:
            self.hand_counts[card.id] += 1
        for card in self.me.discard:
            self.discard_counts[card.id] += 1

    def _scan_main_options(self) -> None:
        if self.context != SelectContext.MAIN:
            return
        for option in self.select.option:
            if option.type == OptionType.PLAY:
                card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)
                if card is None:
                    continue
                if card.id == C.SWITCH:
                    self.can_switch = True
                elif card.id == C.BOSS_ORDERS:
                    self.can_gust = True
            elif option.type == OptionType.EVOLVE:
                card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)
                if card is None:
                    continue
                if card.id == C.HARIYAMA:
                    self.can_gust = True
            elif option.type == OptionType.RETREAT:
                self.can_switch = True
            elif option.type == OptionType.ATTACK:
                self.can_attack = True
                if option.attackId == MEGA_BRAVE:
                    self.can_use_mega_brave = True

    def _my_board(self) -> list[Pokemon | None]:
        return self.me.active + self.me.bench

    def _opponent_board(self) -> list[Pokemon | None]:
        return self.opponent.active + self.opponent.bench

    def _opponent_has(self, ids: set[int]) -> bool:
        return any(pokemon is not None and pokemon.id in ids for pokemon in self._opponent_board())

    def _opponent_is_crustle_wall(self) -> bool:
        return CRUSTLE_AWARE and self._opponent_has({C.DWEBBLE, C.CRUSTLE})

    def _opponent_is_water_deck(self) -> bool:
        return self._opponent_has({C.KYOGRE, C.SNOVER, C.MEGA_ABOMASNOW_EX})

    def _is_ex_attacker(self, pokemon: Pokemon) -> bool:
        data = card_table.get(pokemon.id)
        return bool(data is not None and (getattr(data, "ex", False) or getattr(data, "megaEx", False)))

    def _can_evolve_board_index(self, board_index: int) -> bool:
        for option in self.select.option:
            if option.type != OptionType.EVOLVE:
                continue
            target_index = option.inPlayIndex
            if option.inPlayArea == AreaType.BENCH:
                target_index += 1
            if target_index == board_index:
                return True
        return False

    def _base_attack(self, pokemon: Pokemon, attack_index: int) -> tuple[int, int, int] | None:
        energy_required = 0
        base_damage = 0
        base_score = 0

        if pokemon.id == C.MEGA_LUCARIO_EX:
            if attack_index == 0:
                energy_required = 1
                base_damage = 130
                base_score += 60 * min(3, self.discard_counts[C.BASIC_FIGHTING_ENERGY])
            else:
                energy_required = 2
                base_damage = 270
            if self.my_prizes_left in {2, 3}:
                base_score -= 500
            if self.water_deck and len(self.opponent.prize) <= 3:
                base_score -= 500
        elif attack_index == 1:
            return None
        elif pokemon.id == C.HARIYAMA:
            energy_required = 3
            base_damage = 210
        elif pokemon.id == C.MAKUHITA:
            return None
        elif pokemon.id == C.SOLROCK and self.field_counts[C.LUNATONE] >= 1:
            energy_required = 1
            base_damage = 70

        if base_damage <= 0:
            return None
        return energy_required, base_damage, base_score

    def _base_attack_after_evolution(self, pokemon: Pokemon, board_index: int, attack_index: int):
        if pokemon.id == C.MAKUHITA and attack_index == 0 and self._can_evolve_board_index(board_index):
            return 3, 210, -100
        return self._base_attack(pokemon, attack_index)

    def _plan_attack(self) -> None:
        global plan
        best_score = -1
        plan = AttackPlan()

        if self.state.turn < 2:
            return

        for attacker_index, my_pokemon in enumerate(self._my_board()):
            if my_pokemon is None:
                continue
            if attacker_index != 0 and not self.can_switch:
                break

            for attack_index in range(2):
                attack = self._base_attack_after_evolution(my_pokemon, attacker_index, attack_index)
                if attack is None:
                    continue
                energy_required, base_damage, base_score = attack

                energy_count = len(my_pokemon.energies)
                if attack_index == 1 and attacker_index == 0 and energy_count >= 2 and not self.can_use_mega_brave:
                    break

                needs_energy = False
                if energy_count < energy_required:
                    if self.hand_counts[C.BASIC_FIGHTING_ENERGY] >= 1 and not self.state.energyAttached:
                        energy_count += 1
                        needs_energy = energy_count >= energy_required
                    if not needs_energy:
                        continue

                for target_index, op_pokemon in enumerate(self._opponent_board()):
                    if op_pokemon is None:
                        continue
                    if target_index != 0 and not self.can_gust:
                        break

                    damage = base_damage
                    op_data = card_table.get(op_pokemon.id)
                    if op_data is None:
                        continue
                    if op_data.weakness == EnergyType.FIGHTING:
                        damage *= 2
                    elif op_data.resistance == EnergyType.FIGHTING:
                        damage -= 30

                    crustle_immune = (
                        self.crustle_wall
                        and op_pokemon.id == C.CRUSTLE
                        and self._is_ex_attacker(my_pokemon)
                    )
                    if crustle_immune:
                        damage = 0

                    score = target_score(op_pokemon)
                    prize = prize_count(op_pokemon) if op_pokemon.hp <= damage else 0
                    if prize == 0:
                        score *= damage / max(1, op_pokemon.hp)
                    if len(self.opponent.prize) <= prize:
                        score = 50000

                    if crustle_immune:
                        score = -10000
                    elif self.crustle_wall:
                        if op_pokemon.id == C.CRUSTLE and my_pokemon.id in {C.HARIYAMA, C.MAKUHITA}:
                            score += 2600
                        elif op_pokemon.id == C.CRUSTLE and my_pokemon.id == C.SOLROCK:
                            score += 450
                        active_wall = bool(self.opponent.active and self.opponent.active[0] is not None and self.opponent.active[0].id == C.CRUSTLE)
                        if active_wall and target_index >= 1:
                            score += 700

                    score += base_score
                    score += 220 if attacker_index == 0 else 0
                    score += 300 if target_index == 0 else 0
                    score += energy_count

                    if score > best_score:
                        best_score = score
                        plan = AttackPlan(
                            attacker=attacker_index,
                            target=target_index,
                            attack_index=attack_index,
                            remain_hp=op_pokemon.hp - damage,
                            needs_energy=needs_energy,
                        )

        if self.crustle_wall and _has_attack_plan():
            board = self._my_board()
            targets = self._opponent_board()
            attacker = board[plan.attacker] if 0 <= plan.attacker < len(board) else None
            target = targets[plan.target] if 0 <= plan.target < len(targets) else None
            if isinstance(attacker, Pokemon) and isinstance(target, Pokemon) and target.id == C.CRUSTLE:
                if self._is_ex_attacker(attacker):
                    _DIAG["crustle_final_plan_ex_into_wall"] += 1
                else:
                    _DIAG["crustle_final_plan_non_ex_into_wall"] += 1

    def _energy_target_score(self, pokemon: Pokemon, active: bool) -> int:
        energy_count = len(pokemon.energies)
        score = 8000 + (10 if active else 0)

        if self.crustle_wall:
            if pokemon.id in {C.MAKUHITA, C.HARIYAMA}:
                score += 450
                if len(pokemon.energies) < 3:
                    score += 220
            elif pokemon.id == C.MEGA_LUCARIO_EX:
                score -= 120

        if pokemon.id in {C.MAKUHITA, C.HARIYAMA}:
            score += 1 if pokemon.id == C.HARIYAMA else 0
            score += 100 if energy_count < 3 else 0
            score -= 50 if self.has_ready_hariyama_line else 0
        elif pokemon.id == C.LUNATONE:
            score -= 100
        elif pokemon.id == C.SOLROCK:
            score += 20 if energy_count < 1 else -100
        elif pokemon.id in {C.RIOLU, C.MEGA_LUCARIO_EX}:
            score += 1 if pokemon.id == C.MEGA_LUCARIO_EX else 0
            score += 100 if energy_count < 2 else 0
            score -= 50 if self.has_ready_lucario_line else 0
        return score

    def _score_option(self, option) -> float:
        if option.type == OptionType.NUMBER:
            return option.number
        if option.type == OptionType.YES:
            return 100 if self.context == SelectContext.IS_FIRST else 1
        if option.type == OptionType.NO:
            return 0
        if option.type == OptionType.CARD:
            return self._score_card_choice(option)
        if option.type == OptionType.PLAY:
            return self._score_play(option)
        if option.type == OptionType.ATTACH:
            return self._score_attach(option)
        if option.type == OptionType.EVOLVE:
            return self._score_evolve(option)
        if option.type == OptionType.ABILITY:
            return self._score_ability(option)
        if option.type == OptionType.RETREAT:
            if self.crustle_wall and plan.attacker >= 1:
                return 3200 if _plan_kos() else 2400
            return 2000 if plan.attacker >= 1 else -1
        if option.type == OptionType.ATTACK:
            active = self.me.active[0] if self.me.active else None
            op_active = self.opponent.active[0] if self.opponent.active else None
            if (
                self.crustle_wall
                and isinstance(active, Pokemon)
                and isinstance(op_active, Pokemon)
                and self._is_ex_attacker(active)
                and op_active.id == C.CRUSTLE
                and plan.target < 0
            ):
                _DIAG["crustle_ex_attack_suppressed"] += 1
                return -1
            return 1100 if (option.attackId == MEGA_BRAVE) == (plan.attack_index == 1) else 1000
        return 0

    def _score_card_choice(self, option) -> float:
        card = get_card(self.obs, option.area, option.index, option.playerIndex)
        if card is None:
            return 0

        if self.context in {SelectContext.SWITCH, SelectContext.TO_ACTIVE}:
            return self._score_active_choice(option, card)
        if self.context == SelectContext.SETUP_ACTIVE_POKEMON:
            return self._score_setup_active(card)
        if self.context == SelectContext.SETUP_BENCH_POKEMON:
            return self._score_setup_bench(card)
        if self.context == SelectContext.TO_HAND:
            return self._score_to_hand(card)
        if self.context == SelectContext.TO_BENCH:
            return self._score_to_bench(card)
        if _ctx_is(self.context, "TO_FIELD"):
            return self._score_to_field(option, card)
        if self.context == SelectContext.DISCARD:
            return self._score_discard(card)
        if self.context in {SelectContext.DAMAGE_COUNTER, SelectContext.DAMAGE_COUNTER_ANY}:
            return self._score_damage_counter(option, card)
        if _ctx_is(self.context, "DAMAGE", "EFFECT_TARGET"):
            return self._score_effect_target(option, card)
        if _ctx_is(self.context, "HEAL", "REMOVE_DAMAGE_COUNTER"):
            return self._score_heal_target(option, card)
        if _ctx_is(self.context, "EVOLVES_FROM", "EVOLVES_TO"):
            return self._score_evolution_context(option, card)
        if _ctx_is(self.context, "MULLIGAN"):
            return self._score_mulligan(card)
        if self.context == SelectContext.ATTACH_FROM and isinstance(card, Pokemon):
            return self._energy_target_score(card, option.area == AreaType.ACTIVE)
        return 0

    def _score_active_choice(self, option, card: Pokemon | Card) -> float:
        if not isinstance(card, Pokemon):
            return 0

        if option.playerIndex != self.my_index:
            return 100 if option.index == plan.target - 1 else 0

        score = len(card.energies) * 2
        if option.index == plan.attacker - 1:
            score += 100
        if card.id == C.MEGA_LUCARIO_EX:
            late_ex_liability = self.my_prizes_left in {2, 3} or (self.water_deck and len(self.opponent.prize) <= 3)
            score += 8 if late_ex_liability else 20
            if self.crustle_wall:
                score -= 35
        elif card.id == C.HARIYAMA and len(card.energies) >= 2:
            score += 45 if self.crustle_wall else 15
        elif card.id == C.MAKUHITA and len(card.energies) >= 2:
            score += 35 if self.crustle_wall else 10
        elif card.id == C.SOLROCK:
            score += 5
        elif card.id == C.RIOLU:
            score += 4
        return score

    def _score_setup_active(self, card: Pokemon | Card) -> int:
        if card is None:
            return 0
        if card.id == C.SOLROCK:
            return 2 if self.state.firstPlayer == self.my_index else 4
        if card.id == C.RIOLU:
            return 3
        if card.id == C.MAKUHITA:
            return 1
        return 0

    def _score_setup_bench(self, card: Pokemon | Card) -> int:
        if card is None:
            return 0
        if card.id == C.RIOLU:
            return 120 - 25 * self.field_counts[C.RIOLU]
        if card.id == C.SOLROCK:
            return 90 if self.field_counts[C.SOLROCK] == 0 else -1
        if card.id == C.LUNATONE:
            return 80 if self.field_counts[C.LUNATONE] == 0 else -1
        if card.id == C.MAKUHITA:
            score = 65 if self.field_counts[C.MAKUHITA] == 0 else 10
            if self.crustle_wall:
                score += 120 if self.field_counts[C.MAKUHITA] == 0 else 45
            return score
        return 0

    def _score_to_bench(self, card: Pokemon | Card) -> float:
        if card is None:
            return 0
        data = card_table.get(card.id)
        if data is None or data.cardType != CardType.POKEMON:
            return 0
        return self._score_setup_bench(card)

    def _score_discard(self, card: Pokemon | Card) -> float:
        if card is None:
            return 0
        cid = card.id
        # Positive means "safe/desirable to discard". Required discard contexts will
        # still pick the highest values; optional discard contexts will skip <= 0.
        if self.crustle_wall:
            if cid == C.BASIC_FIGHTING_ENERGY and self.hand_counts[cid] <= 2:
                return -120
            if cid in {C.MAKUHITA, C.HARIYAMA, C.SWITCH, C.GRAVITY_MOUNTAIN}:
                return -90
        if cid == C.BASIC_FIGHTING_ENERGY:
            score = 45 if self.hand_counts[cid] >= 2 else 5
            if plan.needs_energy and not self.state.energyAttached:
                score -= 200
            return score
        if self.hand_counts[cid] >= 2:
            return 70
        if cid in {C.LUNATONE, C.SOLROCK} and self.field_counts[cid] >= 1:
            return 55
        if cid == C.GRAVITY_MOUNTAIN and self.stadium_id == C.GRAVITY_MOUNTAIN:
            return 50
        if cid in {C.CARMINE, C.LILLIE_DETERMINATION} and self.state.supporterPlayed:
            return 30
        if cid == C.MEGA_LUCARIO_EX and self.field_counts[C.RIOLU] == 0:
            return -80
        if cid == C.HARIYAMA and self.field_counts[C.MAKUHITA] == 0:
            return -50
        if cid in {C.RIOLU, C.MAKUHITA, C.BOSS_ORDERS, C.HERO_CAPE}:
            return -40
        return 0

    def _is_opponent_option(self, option) -> bool:
        return getattr(option, "playerIndex", self.my_index) == self.op_index

    def _score_damage_counter(self, option, card: Pokemon | Card) -> float:
        if not isinstance(card, Pokemon):
            return 0
        if self._is_opponent_option(option):
            return 10000 + prize_count(card) * 1000 - getattr(card, "hp", 0) + _damage_on(card) * 5
        return -target_score(card)

    def _score_effect_target(self, option, card: Pokemon | Card) -> float:
        if not isinstance(card, Pokemon):
            return self._score_to_hand(card)
        if self._is_opponent_option(option):
            return 2000 + target_score(card) + _damage_on(card) * 8
        score = 300 + len(card.energies) * 50 + len(card.tools) * 40 + _damage_on(card) * 8
        if card.id == C.MEGA_LUCARIO_EX:
            score += 250
        elif card.id in {C.RIOLU, C.HARIYAMA}:
            score += 120
        elif card.id in {C.SOLROCK, C.LUNATONE, C.MAKUHITA}:
            score += 70
        return score

    def _score_heal_target(self, option, card: Pokemon | Card) -> float:
        if not isinstance(card, Pokemon):
            return 0
        damage = _damage_on(card)
        if self._is_opponent_option(option):
            return -1000 - damage
        score = damage * 20 + len(card.energies) * 30 + len(card.tools) * 25
        if card.id == C.MEGA_LUCARIO_EX:
            score += 300
        elif card.id in {C.RIOLU, C.HARIYAMA}:
            score += 120
        return score if damage > 0 else max(0, score // 10)

    def _score_evolution_context(self, option, card: Pokemon | Card) -> float:
        if not isinstance(card, Pokemon):
            return 0
        if self._is_opponent_option(option):
            return target_score(card)
        if card.id in {C.RIOLU, C.MEGA_LUCARIO_EX}:
            return 900 + len(card.energies) * 30
        if card.id in {C.MAKUHITA, C.HARIYAMA}:
            return 650 + len(card.energies) * 30
        return 100 + len(card.energies) * 20

    def _score_to_field(self, option, card: Pokemon | Card) -> float:
        if isinstance(card, Pokemon):
            return self._score_to_bench(card) + self._score_setup_active(card)
        return self._score_to_hand(card)

    def _score_mulligan(self, card: Pokemon | Card) -> float:
        if not isinstance(card, Pokemon):
            return 0
        data = card_table.get(card.id)
        if data is None:
            return 0
        is_basic = (
            getattr(data, "basic", False)
            or (
                data.cardType == CardType.POKEMON
                and not getattr(data, "stage1", False)
                and not getattr(data, "stage2", False)
                and not getattr(data, "megaEx", False)
            )
        )
        if not is_basic:
            return 0
        return self._score_setup_active(card) + self._score_setup_bench(card)

    def _score_to_hand(self, card: Pokemon | Card) -> float:
        if card is None:
            return 0
        score = 200 - self.hand_counts[card.id] * 100
        if self.crustle_wall:
            if card.id == C.MAKUHITA:
                score += 120 if self.field_counts[C.MAKUHITA] == 0 else 30
            elif card.id == C.HARIYAMA:
                score += 140 if self.field_counts[C.MAKUHITA] >= 1 else 25
            elif card.id == C.SWITCH:
                score += 80
            elif card.id in {C.POKE_PAD, C.FIGHTING_GONG}:
                score += 55
            elif card.id == C.GRAVITY_MOUNTAIN:
                score += 80
            elif card.id == C.BASIC_FIGHTING_ENERGY:
                score += 45
            elif card.id == C.BOSS_ORDERS:
                score += 60
        if card.id == C.MAKUHITA:
            score += -10 if self.field_counts[card.id] >= 1 else 10
        elif card.id == C.HARIYAMA:
            score += 20 if self.field_counts[C.MAKUHITA] >= 1 else -20
        elif card.id == C.LUNATONE:
            score += -250 if self.field_counts[card.id] >= 1 else 60
        elif card.id == C.SOLROCK:
            score += -250 if self.field_counts[card.id] >= 1 else 50
        elif card.id == C.RIOLU:
            lucario_line = self.field_counts[C.RIOLU] + self.field_counts[C.MEGA_LUCARIO_EX]
            score += -150 if lucario_line >= 2 else -3 if lucario_line >= 1 else 40
        elif card.id == C.MEGA_LUCARIO_EX:
            score += 40 if self.field_counts[C.RIOLU] >= 1 else -15
        elif card.id == C.BASIC_FIGHTING_ENERGY:
            score += 30 if not ability_used or not self.state.energyAttached else -1
        return score

    def _score_play(self, option) -> float:
        card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)
        if card is None:
            return 0
        data = card_table.get(card.id)
        if data is None:
            return 0
        if data.cardType == CardType.POKEMON:
            return self._score_play_pokemon(card)
        return self._score_play_trainer(card)

    def _score_play_pokemon(self, card: Card) -> float:
        score = 20000
        if card.id in {C.LUNATONE, C.SOLROCK} and self.field_counts[card.id] >= 1:
            return -1
        if card.id == C.RIOLU and self.field_counts[C.RIOLU] + self.field_counts[C.MEGA_LUCARIO_EX] >= 2:
            return -1
        return score

    def _score_play_trainer(self, card: Card) -> float:
        if _plan_kos() and card.id in {
            C.DUSK_BALL,
            C.FIGHTING_GONG,
            C.POKE_PAD,
            C.CARMINE,
            C.LILLIE_DETERMINATION,
            C.GRAVITY_MOUNTAIN,
        }:
            return -1
        if card.id == C.SWITCH:
            if plan.attacker > 0 and _plan_kos():
                return 14000
            return 6500 if plan.attacker > 0 else -1
        if card.id == C.PREMIUM_POWER_PRO:
            if self.state.supporterPlayed and _plan_kos():
                return -1
            if not self.can_attack:
                can_bridge_draw = (
                    not self.state.supporterPlayed
                    and self.hand_counts[C.CARMINE] > 0
                    and self.hand_counts[C.LILLIE_DETERMINATION] == 0
                    and not self._low_deck()
                )
                return 3050 if can_bridge_draw else -1
            return 5000
        if card.id == C.BOSS_ORDERS:
            if plan.target >= 1 and _plan_kos():
                return 15000
            if self.crustle_wall and plan.target >= 1:
                return 7600
            return 4200 if plan.target >= 1 else -1
        if card.id == C.CARMINE:
            return -1 if self._low_deck() else 3000
        if card.id == C.LILLIE_DETERMINATION:
            return -1 if self._low_deck() else 3100
        if card.id == C.GRAVITY_MOUNTAIN:
            return self._score_gravity_mountain()
        return 10000

    def _score_gravity_mountain(self) -> float:
        if self.crustle_wall and self.stadium_id != C.GRAVITY_MOUNTAIN:
            return 3600
        opponent_has_stage2 = any(
            pokemon is not None and card_table.get(pokemon.id) is not None and card_table[pokemon.id].stage2
            for pokemon in self._opponent_board()
        )
        if opponent_has_stage2:
            return 3500
        return 1200 if self.stadium_id else -1

    def _low_deck(self) -> bool:
        return self.me.deckCount <= LOW_DECK_COUNT

    def _score_attach(self, option) -> float:
        card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)
        pokemon = get_card(self.obs, option.inPlayArea, option.inPlayIndex, self.my_index)
        if card is None or not isinstance(pokemon, Pokemon):
            return 0

        if card.id == C.HERO_CAPE:
            if self.water_deck and not self.crustle_wall:
                if pokemon.id == C.RIOLU:
                    return 12200
                if pokemon.id == C.MEGA_LUCARIO_EX:
                    return 12800
            score = 7000
            if self.crustle_wall:
                if pokemon.id in {C.MAKUHITA, C.HARIYAMA}:
                    score += 350
                elif pokemon.id == C.MEGA_LUCARIO_EX:
                    score -= 200
            if pokemon.id == C.RIOLU:
                score += 100
            elif pokemon.id == C.MEGA_LUCARIO_EX:
                score += 200
            return score

        score = self._energy_target_score(pokemon, option.inPlayArea == AreaType.ACTIVE)
        board_index = option.inPlayIndex if option.inPlayArea == AreaType.ACTIVE else option.inPlayIndex + 1
        if board_index == plan.attacker and plan.needs_energy:
            score += 200
        return score

    def _score_evolve(self, option) -> float:
        pokemon = get_card(self.obs, option.inPlayArea, option.inPlayIndex, self.my_index)
        if not isinstance(pokemon, Pokemon):
            return 0
        if pokemon.id == C.MAKUHITA and plan.target == 0 and not self.crustle_wall:
            return -1
        evolve_card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)
        score = 9000 + len(pokemon.energies)
        if evolve_card is not None:
            if evolve_card.id == C.MEGA_LUCARIO_EX and pokemon.id == C.RIOLU:
                score += 350
            elif evolve_card.id == C.HARIYAMA and pokemon.id == C.MAKUHITA:
                score += 180
                if self.crustle_wall:
                    score += 650
        return score

    def _score_ability(self, option) -> float:
        card = get_card(self.obs, option.area, option.index, self.my_index)
        if card is None:
            return 0
        if card.id == C.LUMIOSE_CITY:
            return 1
        if card.id == C.LUNATONE and self._low_deck():
            return -1
        return 30000

    def remember_chosen_side_effects(self, selection: list[int]) -> None:
        global ability_used
        if self.context != SelectContext.MAIN:
            return
        for idx in selection:
            if not (0 <= idx < len(self.select.option)):
                continue
            option = self.select.option[idx]
            if option.type != OptionType.ABILITY:
                continue
            card = get_card(self.obs, option.area, option.index, self.my_index)
            if card is not None and card.id == C.LUNATONE:
                ability_used = True
'''

In [9]:
MAIN_AGENT_ENTRYPOINT = r'''def agent(obs_dict: dict) -> list[int]:
    global pre_turn, ability_used, plan

    # Deck-selection phase: the engine asks for the deck with select == None.
    # Return the 60-card id list (different return semantics from a normal turn).
    try:
        select_is_none = isinstance(obs_dict, dict) and obs_dict.get("select") is None
    except Exception:
        select_is_none = False
    if select_is_none:
        _DIAG["deck_returns"] += 1
        return my_deck

    _DIAG["decisions"] += 1
    try:
        obs = to_observation_class(obs_dict)
        if obs.select is None:
            _DIAG["deck_returns"] += 1
            _DIAG["decisions"] -= 1
            return my_deck

        if obs.current is not None and pre_turn != obs.current.turn:
            pre_turn = obs.current.turn
            ability_used = False
            plan = AttackPlan()

        try:
            policy = LucarioPolicy(obs)
            _diag_observe(obs)
            ranked, scores = policy.rank()
            planned = search_plan(obs_dict, obs, ranked, scores) if USE_SEARCH else None
            ranked_use = ranked if planned is None else planned
            selection = normalize_selection(ranked_use, scores, obs.select)
            policy.remember_chosen_side_effects(selection)
            _diag_observe_choice(obs, selection)
            _DIAG["policy_ok"] += 1
            return selection
        except Exception as exc:
            _diag_record_error(exc)
            _DIAG["policy_fallback"] += 1
            return _legal_fallback(obs.select)
    except Exception as exc:
        _diag_record_error(exc)
        _DIAG["obs_fallback"] += 1
        return _legal_fallback_from_dict(obs_dict if isinstance(obs_dict, dict) else {})
'''

In [10]:
MAIN_SECTION_REGISTRY = [
    ("imports, constants, and deck loading", MAIN_PRELUDE),
    ("diagnostics", MAIN_DIAGNOSTICS),
    ("optional forward search", MAIN_FORWARD_SEARCH),
    ("selection normalization", MAIN_SELECTION_HELPERS),
    ("card and target helpers", MAIN_CARD_HELPERS),
    ("Lucario policy", MAIN_POLICY),
    ("agent entrypoint", MAIN_AGENT_ENTRYPOINT),
]
MAIN_SECTIONS = [source for _, source in MAIN_SECTION_REGISTRY]
MAIN_CODE = "\n\n".join(section.rstrip() for section in MAIN_SECTIONS)

import re
MAIN_CODE, embedded_deck_replacements = re.subn(
    r"EMBEDDED_DECK = \[\n(?:    [0-9, ]+\n)+\]",
    "EMBEDDED_DECK = " + repr(DECK),
    MAIN_CODE,
    count=1,
)
assert embedded_deck_replacements == 1, "EMBEDDED_DECK replacement failed"
MAIN_CODE = MAIN_CODE.replace("USE_SEARCH = False", f"USE_SEARCH = {USE_SEARCH_IN_AGENT}", 1)
MAIN_CODE = MAIN_CODE.replace("CRUSTLE_AWARE = True", f"CRUSTLE_AWARE = {CRUSTLE_AWARE_IN_AGENT}", 1)
MAIN_CODE = MAIN_CODE.replace("SEARCH_NODE_BUDGET = 120", f"SEARCH_NODE_BUDGET = {SEARCH_NODE_BUDGET}", 1)
MAIN_CODE = MAIN_CODE.replace("SEARCH_TIME_BUDGET_S = 1.5", f"SEARCH_TIME_BUDGET_S = {SEARCH_TIME_BUDGET_S}", 1)
MAIN_CODE = MAIN_CODE.replace("SEARCH_ACTION_CAP = 20", f"SEARCH_ACTION_CAP = {SEARCH_ACTION_CAP}", 1)

(BUILD_DIR / "main.py").write_text(MAIN_CODE + "\n")
print(
    "wrote", BUILD_DIR / "main.py",
    "bytes=", len(MAIN_CODE),
    "| sections=", ", ".join(name for name, _ in MAIN_SECTION_REGISTRY),
    "| USE_SEARCH=", USE_SEARCH_IN_AGENT,
    "| node_budget=", SEARCH_NODE_BUDGET,
)


wrote /kaggle/working/phase1_lucario_v2_build/main.py bytes= 53652 | sections= imports, constants, and deck loading, diagnostics, optional forward search, selection normalization, card and target helpers, Lucario policy, agent entrypoint | USE_SEARCH= False | node_budget= 120


In [11]:
main_text = (BUILD_DIR / "main.py").read_text()
compile(main_text, str(BUILD_DIR / "main.py"), "exec")
assert "def agent(obs_dict: dict)" in main_text
assert "import random" in main_text
assert "EMBEDDED_DECK = " + repr(DECK) in main_text
assert "def normalize_selection" in main_text
assert "USE_SEARCH = False" in main_text
assert "def search_plan" in main_text
assert "score > 0" in main_text
assert "_resolve_deck_path" in main_text
assert "remember_chosen_side_effects" in main_text
assert "search_begin_input" in main_text
assert "search_release(sid)" in main_text
assert "def _rollout_search_candidate" in main_text
assert "def _should_search" in main_text
assert "SearchState" not in main_text
assert "search_end()" not in main_text
assert "def _pokemon_max_hp" in main_text
assert "def _ctx_is" in main_text
assert "maxHp" in main_text
assert "def _score_effect_target" in main_text
assert "def _score_heal_target" in main_text
assert "def _score_evolution_context" in main_text
assert "def _score_mulligan" in main_text
assert "def _diag_record_search_error" in main_text
assert "search_errors" in main_text
assert "_score_setup_bench" in main_text
assert "return 3600" in main_text
assert "crustle_immune" in main_text
assert "def _opponent_is_crustle_wall" in main_text
assert "def _opponent_is_water_deck" in main_text
assert "pokemon.id == C.SNOVER" in main_text
assert "pokemon.id == C.RIOLU" in main_text
assert "return 12800" in main_text
assert "def _has_attack_plan" in main_text
assert "def _plan_kos" in main_text
assert "if _plan_kos():" in main_text
assert "yes_no_contexts" in main_text
assert "plan.remain_hp <= 0 and card.id in" not in main_text
assert "plan.attacker > 0 and plan.remain_hp <= 0" not in main_text
assert "self.state.supporterPlayed and plan.remain_hp <= 0" not in main_text
assert "plan.target >= 1 and plan.remain_hp <= 0" not in main_text
assert "getattr(pokemon, \"appearThisTurn\", False)" not in main_text
assert "C.MEGA_ABOMASNOW_EX" in main_text
assert "late_ex_liability" in main_text
assert "and not self.crustle_wall" in main_text
assert "op_active.id == C.CRUSTLE" in main_text
assert "C.CRUSTLE" in main_text
assert "CRUSTLE_AWARE = True" in main_text
assert "crustle_seen" in main_text
assert "crustle_wall_policy_seen" in main_text
assert "crustle_ex_attack_suppressed" in main_text
assert "crustle_final_plan_non_ex_into_wall" in main_text
assert "submission.csv" not in main_text
print("main.py syntax and required hooks: OK")

main.py syntax and required hooks: OK


## Diagnostics And Evaluation

The diagnostics answer three questions:

1. Did the policy run, or did it silently fall back?
2. Which legal actions and attack IDs were actually offered and chosen?
3. Do matchup probes show the intended branch firing?

The Crustle probe reports OFF/ON win-rate deltas for the active deck and the Crustle-retuned branch. The local-safe path skips engine probes when `cg` is unavailable; on Kaggle, the same cells run mirror, random, search audit, and Crustle wall checks.

In [12]:
def iter_cg_candidates():
    explicit = [
        ROOT / "cg",
        Path("/kaggle/working/cg"),
        Path("/kaggle/input/pokemon-tcg-ai-battle/cg"),
        Path("/kaggle/input/pokemon-tcg-ai-battle-simulation/cg"),
    ]
    patterns = [
        "/kaggle/input/**/cg-lib/cg",
        "/kaggle/input/**/cg",
    ]
    seen = set()
    for p in explicit:
        if p not in seen:
            seen.add(p)
            yield p
    for pattern in patterns:
        for match in glob.glob(pattern, recursive=True):
            p = Path(match)
            if p not in seen:
                seen.add(p)
                yield p


def copy_cg_package():
    for src in iter_cg_candidates():
        if src.exists() and src.is_dir() and (src / "api.py").exists():
            dst = BUILD_DIR / "cg"
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "*.pyo"))
            print("copied cg package from", src)
            return dst
    print("cg package not found in the current runtime.")
    return None

CG_DIR = copy_cg_package()




copied cg package from /kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission/cg


In [13]:
# ============================================================================
# Shared helpers used by both the EDA cells and the validation gate.
# ============================================================================
def load_deck_ids():
    return [int(x.strip()) for x in (BUILD_DIR / "deck.csv").read_text().splitlines() if x.strip()]


def random_legal_agent_factory(deck):
    """A legal-but-random opponent: returns maxCount distinct legal option indices."""
    def random_legal_agent(obs_dict):
        sel = obs_dict.get("select")
        if sel is None:
            return deck
        options = sel.get("option") or []
        n = len(options)
        minc = max(0, min(sel.get("minCount", 0), n))
        maxc = max(minc, min(sel.get("maxCount", minc), n))
        return random.sample(range(n), maxc) if maxc else []
    return random_legal_agent


def import_generated_module():
    """Import the freshly-written main.py as a module so we can read its _DIAG."""
    import importlib.util
    if str(BUILD_DIR) not in sys.path:
        sys.path.insert(0, str(BUILD_DIR))
    spec = importlib.util.spec_from_file_location("phase1_generated_main", BUILD_DIR / "main.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


In [14]:
def run_optional_engine_eda():
    if CG_DIR is None:
        print("skip engine EDA: cg/ unavailable in this runtime")
        return None

    if str(BUILD_DIR) not in sys.path:
        sys.path.insert(0, str(BUILD_DIR))

    try:
        from cg.api import CardType, to_observation_class, all_card_data
        from cg.game import battle_start, battle_finish
    except Exception as e:
        print("skip engine EDA: cg import failed:", repr(e))
        return None

    cards = all_card_data()
    card_rows = []
    for card in cards:
        card_rows.append({
            "card_id": getattr(card, "cardId", None),
            "name": getattr(card, "name", None),
            "card_type": str(getattr(card, "cardType", None)).split(".")[-1],
            "hp": getattr(card, "hp", None),
            "ex": getattr(card, "ex", None),
            "megaEx": getattr(card, "megaEx", None),
        })

    if pd is not None:
        card_df = pd.DataFrame(card_rows)
        display(card_df.groupby("card_type", as_index=False).size())
        display(card_df[["hp"]].describe())
    else:
        print("total cards:", len(card_rows))

    deck = [int(x.strip()) for x in (BUILD_DIR / "deck.csv").read_text().splitlines() if x.strip()]
    obs_raw, start_data = battle_start(deck, deck)
    try:
        if obs_raw is None:
            print("battle_start failed:", start_data)
            return None

        obs = to_observation_class(obs_raw) if isinstance(obs_raw, dict) else obs_raw
        print("battle_start: OK")
        print("obs.current is None:", getattr(obs, "current", None) is None)
        print("obs.select is None:", getattr(obs, "select", None) is None)
        if getattr(obs, "select", None) is not None:
            print("select context:", getattr(obs.select, "context", None))
            print("select.option count:", len(obs.select.option))
            print("min/max:", obs.select.minCount, obs.select.maxCount)
            print("option type sample:", [getattr(o, "type", None) for o in obs.select.option[:10]])
        return obs
    finally:
        try:
            battle_finish()
        except Exception:
            pass

ENGINE_EDA_OBS = run_optional_engine_eda()





,card_type,size
0,0,1056
1,1,77
2,2,27
3,3,61
4,4,26
5,5,8
6,6,12


,hp
count,1267.000000
mean,102.257301
std,78.495651
min,0.000000
25%,60.000000
50%,90.000000
75%,130.000000
max,380.000000


battle_start: OK
obs.current is None: False
obs.select is None: False
select context: 41
select.option count: 2
min/max: 1 1
option type sample: [1, 2]


In [15]:
# Curated from the four official sample decks (card ids + counts). Used for a
# structure comparison only - no engine required.
SAMPLE_ARCHETYPES = {
    "Mega Lucario ex (Fighting)": {
        "axis": "Fighting multi-attacker; Riolu->Mega Lucario ex core, Hariyama/Solrock backup",
        "pokemon": [(673, 2), (674, 2), (675, 2), (676, 3), (677, 3), (678, 4)],
        "trainer": [(1102, 4), (1123, 2), (1141, 4), (1142, 4), (1152, 4), (1159, 1),
                     (1182, 2), (1192, 4), (1227, 4), (1252, 2)],
        "energy": [(6, 13)],
        "prize_liability": [("Mega Lucario ex", 3)],
    },
    "Dragapult ex (Fire/Psychic spread)": {
        "axis": "Stage-2 spread/snipe; Crushing Hammer disruption; 2-color",
        "pokemon": [(119, 4), (120, 4), (121, 3), (140, 1), (184, 1), (235, 2), (1071, 1)],
        "trainer": [(1079, 2), (1080, 1), (1086, 4), (1097, 2), (1120, 4), (1121, 4),
                     (1152, 3), (1156, 1), (1182, 3), (1198, 4), (1210, 2), (1227, 4), (1256, 2)],
        "energy": [(2, 4), (5, 4)],
        "prize_liability": [("Dragapult ex", 2), ("Fezandipiti ex", 2),
                             ("Latias ex", 2), ("Meowth ex", 2)],
    },
    "Iono Bellibolt ex (Lightning)": {
        "axis": "Lightning engine; Bellibolt ex acceleration; Iono support package",
        "pokemon": [(265, 3), (268, 3), (269, 3), (270, 3), (271, 3)],
        "trainer": [(1086, 3), (1097, 2), (1110, 1), (1118, 1), (1121, 3), (1152, 2),
                     (1227, 4), (1233, 4), (1254, 3)],
        "energy": [(4, 22)],
        "prize_liability": [("Bellibolt ex", 2)],
    },
    "Mega Abomasnow ex (Water)": {
        "axis": "Water beatdown; minimal tech; 34 basic energy",
        "pokemon": [(721, 2), (722, 4), (723, 4)],
        "trainer": [(1121, 4), (1126, 1), (1192, 4), (1227, 4), (1262, 3)],
        "energy": [(3, 34)],
        "prize_liability": [("Mega Abomasnow ex", 3), ("Kyogre", 1)],
    },
}


def _sum_counts(pairs):
    return sum(c for _, c in pairs)


arch_rows = []
for name, d in SAMPLE_ARCHETYPES.items():
    pk, tr, en = _sum_counts(d["pokemon"]), _sum_counts(d["trainer"]), _sum_counts(d["energy"])
    max_prize = max((p for _, p in d["prize_liability"]), default=1)
    n_two_prize_plus = sum(1 for _, p in d["prize_liability"] if p >= 2)
    arch_rows.append({
        "archetype": name,
        "pokemon": pk, "trainer": tr, "energy": en, "total": pk + tr + en,
        "unique_pokemon_lines": len(d["pokemon"]),
        "max_prize_given_up": max_prize,
        "ex_megaex_lines": n_two_prize_plus,
        "axis": d["axis"],
    })

if pd is not None:
    arch_df = pd.DataFrame(arch_rows)
    display(arch_df[["archetype", "pokemon", "trainer", "energy", "total",
                     "unique_pokemon_lines", "max_prize_given_up", "ex_megaex_lines"]])
else:
    for r in arch_rows:
        print(r)

print()
print("Takeaways:")
print("- Lucario is the field default -> mirror-heavy; edge there comes mostly from not crashing.")
print("- Abomasnow is the simplest shell (1 attacker line, 34 energy) -> good fixed opponent for eval.")
print("- Dragapult carries the most prize liability (4 two-prize+ lines) but has spread/disruption upside.")
print("- Bellibolt is the leanest engine deck. For Phase-2 differentiation, the non-Lucario space is wide open.")


,archetype,pokemon,trainer,energy,total,unique_pokemon_lines,max_prize_given_up,ex_megaex_lines
0,Mega Lucario ex (Fighting),16,31,13,60,6,3,1
1,Dragapult ex (Fire/Psychic spread),16,36,8,60,7,2,4
2,Iono Bellibolt ex (Lightning),15,23,22,60,5,2,1
3,Mega Abomasnow ex (Water),10,16,34,60,3,3,1



Takeaways:
- Lucario is the field default -> mirror-heavy; edge there comes mostly from not crashing.
- Abomasnow is the simplest shell (1 attacker line, 34 energy) -> good fixed opponent for eval.
- Dragapult carries the most prize liability (4 two-prize+ lines) but has spread/disruption upside.
- Bellibolt is the leanest engine deck. For Phase-2 differentiation, the non-Lucario space is wide open.


In [16]:
def run_action_space_eda(n_games=5, max_states=4000):
    """Roll out a few random-vs-random games and tally the action space + hidden info."""
    if CG_DIR is None:
        print("skip action-space EDA: cg/ unavailable in this runtime.")
        return None
    try:
        from cg.game import battle_start, battle_select, battle_finish
    except Exception as e:
        print("skip action-space EDA: cg import failed:", repr(e))
        return None

    from collections import Counter
    deck = load_deck_ids()
    rnd = random_legal_agent_factory(deck)
    opt_types, contexts = Counter(), Counter()
    option_lens, max_counts = [], []
    hidden_checked = False
    states = 0

    for _ in range(n_games):
        try:
            obs, start = battle_start(deck, deck)
        except Exception as e:
            print("battle_start failed:", repr(e))
            break
        if getattr(start, "errorPlayer", -1) is not None and getattr(start, "errorPlayer", -1) >= 0:
            battle_finish()
            continue
        steps = 0
        while obs["current"]["result"] < 0 and states < max_states:
            sel = obs.get("select")
            if sel is not None:
                opts = sel.get("option") or []
                option_lens.append(len(opts))
                max_counts.append(sel.get("maxCount"))
                contexts[str(sel.get("context"))] += 1
                for o in opts:
                    opt_types[str(o.get("type"))] += 1
                states += 1
            if not hidden_checked and obs.get("current") is not None:
                try:
                    cur = obs["current"]
                    players = cur.get("players", [])
                    if len(players) == 2:
                        me = cur.get("yourIndex", 0)
                        opp = players[1 - me]
                        print("Hidden-information check (grounds Phase-2 belief modeling):")
                        print("  opponent handCount   :", opp.get("handCount"))
                        print("  opponent 'hand' field:", "present" if opp.get("hand") else "absent/empty (count-only)")
                        print("  my prize entries     :", players[me].get("prize", [])[:3], "(face-down should be None)")
                        hidden_checked = True
                except Exception as e:
                    print("  hidden-info check skipped:", repr(e))
                    hidden_checked = True
            obs = battle_select(rnd(obs))
            steps += 1
            if steps > MAX_STEPS_PER_GAME:
                break
        battle_finish()

    print()
    print("sampled decision states:", states)
    print("OptionType frequency   :", dict(opt_types.most_common()))
    print("SelectContext frequency:", dict(contexts.most_common()))
    if pd is not None and option_lens:
        display(pd.Series(option_lens).describe().to_frame("len(option) per decision"))
        clean_max = [m for m in max_counts if m is not None]
        if clean_max:
            display(pd.Series(clean_max).describe().to_frame("maxCount per decision"))
    return {"opt_types": dict(opt_types), "contexts": dict(contexts), "states": states}


ACTION_SPACE_EDA = run_action_space_eda(n_games=5)


Hidden-information check (grounds Phase-2 belief modeling):
  opponent handCount   : 0
  opponent 'hand' field: absent/empty (count-only)
  my prize entries     : [] (face-down should be None)

sampled decision states: 678
OptionType frequency   : {'7': 1190, '8': 1036, '3': 695, '14': 480, '13': 243, '9': 151, '12': 149, '10': 69, '6': 51, '1': 8, '2': 8, '0': 3}
SelectContext frequency: {'0': 480, '7': 60, '3': 45, '30': 23, '21': 18, '8': 11, '1': 10, '4': 9, '22': 9, '41': 5, '2': 4, '43': 3, '38': 1}


,len(option) per decision
count,678.000000
mean,6.022124
std,4.322773
min,1.000000
25%,3.000000
50%,5.000000
75%,8.000000
max,32.000000


,maxCount per decision
count,678.000000
mean,1.028024
std,0.225637
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,3.000000


In [17]:
def validate_submission_bundle(path: Path, require_cg: bool = True) -> list[str]:
    path = Path(path)
    assert path.exists(), f"missing archive: {path}"
    assert path.name.endswith(".tar.gz"), path

    with tarfile.open(path, "r:gz") as tar:
        names = tar.getnames()
        name_set = set(names)

        errors = []
        if "main.py" not in name_set:
            errors.append("main.py must be at archive root")
        if "deck.csv" not in name_set:
            errors.append("deck.csv must be at archive root")
        if any(name.endswith("submission.csv") or name == "submission.csv" for name in names):
            errors.append("submission.csv should not be included for this agent competition")
        forbidden = [n for n in names if n.endswith((".ipynb", ".pyc", ".pyo")) or "__pycache__" in n or n.startswith("benchmark_logs/")]
        if forbidden:
            errors.append(f"forbidden artifacts included: {forbidden[:5]}")

        if "deck.csv" in name_set:
            raw = tar.extractfile("deck.csv").read().decode().splitlines()
            deck = [int(line.strip()) for line in raw if line.strip()]
            if len(deck) != 60:
                errors.append(f"deck.csv must contain 60 integer IDs, got {len(deck)}")

        if "main.py" in name_set:
            main = tar.extractfile("main.py").read().decode()
            compile(main, "main.py", "exec")
            if "def agent" not in main:
                errors.append("main.py must expose agent(obs_dict)")
            if "from cg.api" in main and require_cg:
                required_cg = {"cg/api.py", "cg/game.py"}
                missing_cg = sorted(required_cg - name_set)
                has_binary = any(n in name_set for n in ["cg/libcg.so", "cg/cg.dll"])
                if missing_cg:
                    errors.append(f"missing cg files: {missing_cg}")
                if not has_binary:
                    errors.append("missing cg binary: expected cg/libcg.so or cg/cg.dll")

        if errors:
            raise AssertionError("\n".join(errors))

    print("submission contract OK")
    print("archive files:", len(names))
    for name in sorted(names)[:20]:
        print(" -", name)
    if len(names) > 20:
        print(" ...")
    return names




In [18]:
# ============================================================================
# Phase-1 VALIDATION GATE  (battle_start based, with fallback diagnostics)
# ----------------------------------------------------------------------------
# Why battle_start and not the env wrapper: it runs in-process (so we can read the
# agent's _DIAG), gives an unambiguous winner index, and matches the official
# samples. The gate checks engine errors, win-rate, policy fallback, and search
# failure rate when search is enabled. A policy can be stable while search is
# silently failing, so these signals are tracked separately.
# ============================================================================
def play_game(battle_start, battle_select, battle_finish, deck0, deck1, agent0, agent1):
    """Return (result, steps, error). result: 2 draw / 0|1 winner seat / None on error."""
    try:
        obs, start = battle_start(deck0, deck1)
    except Exception as e:
        return None, 0, "battle_start raised: " + repr(e)
    err = getattr(start, "errorPlayer", -1)
    if err is not None and err >= 0:
        try:
            battle_finish()
        except Exception:
            pass
        return None, 0, "deck error: player=%s type=%s" % (err, getattr(start, "errorType", "?"))
    steps = 0
    try:
        while obs["current"]["result"] < 0:
            seat = obs["current"]["yourIndex"]
            sel = (agent0 if seat == 0 else agent1)(obs)
            obs = battle_select(sel)
            steps += 1
            if steps > MAX_STEPS_PER_GAME:
                battle_finish()
                return None, steps, "exceeded MAX_STEPS_PER_GAME"
        result = obs["current"]["result"]
    except Exception as e:
        try:
            battle_finish()
        except Exception:
            pass
        return None, steps, "game loop raised: " + repr(e)
    battle_finish()
    return result, steps, None


def run_validation_gate(n_games, enforce=True):
    if CG_DIR is None:
        print("skip validation gate: cg/ unavailable in this runtime.")
        return None
    try:
        from cg.game import battle_start, battle_select, battle_finish
    except Exception as e:
        print("skip validation gate: cg import failed:", repr(e))
        return None

    deck = load_deck_ids()
    mod = import_generated_module()
    agent = mod.agent
    rnd = random_legal_agent_factory(deck)
    report = {}

    # ---- Mirror: agent vs a copy of itself (what the ladder validation does). ----
    mod.diag_reset()
    mirror_errors = 0
    for g in range(n_games):
        res, steps, err = play_game(battle_start, battle_select, battle_finish, deck, deck, agent, agent)
        if err is not None or res is None:
            mirror_errors += 1
            if mirror_errors <= 5:
                print("  mirror engine error:", err)
    md = mod.diag_snapshot()
    report["mirror_errors"] = mirror_errors
    report["mirror_fallback_rate"] = md["fallback_rate"]
    report["mirror_search_fail_rate"] = md.get("search_fail_rate", 0.0)
    report["mirror_decisions"] = md["decisions"]
    report["mirror_top_errors"] = sorted(md.get("policy_errors", md["errors"]).items(), key=lambda kv: -kv[1])[:6]
    report["mirror_top_search_errors"] = sorted(md.get("search_errors", {}).items(), key=lambda kv: -kv[1])[:6]

    # ---- vs legal-random, alternating seat. ----
    mod.diag_reset()
    w = d = l = 0
    for g in range(n_games):
        if g % 2 == 0:
            res, steps, err = play_game(battle_start, battle_select, battle_finish, deck, deck, agent, rnd)
            my_seat = 0
        else:
            res, steps, err = play_game(battle_start, battle_select, battle_finish, deck, deck, rnd, agent)
            my_seat = 1
        if err is not None or res is None:
            continue
        if res == 2:
            d += 1
        elif res == my_seat:
            w += 1
        else:
            l += 1
    played = w + d + l
    winrate = w / played if played else 0.0
    rd = mod.diag_snapshot()
    report["vs_random_w_d_l"] = (w, d, l)
    report["vs_random_winrate"] = winrate
    report["vs_random_fallback_rate"] = rd["fallback_rate"]
    report["vs_random_search_fail_rate"] = rd.get("search_fail_rate", 0.0)
    report["vs_random_top_search_errors"] = sorted(rd.get("search_errors", {}).items(), key=lambda kv: -kv[1])[:6]

    # ---- Report ----
    print("=" * 64)
    print("PHASE-1 VALIDATION GATE   (n_games per matchup = %d)" % n_games)
    print("-" * 64)
    print("Mirror     : errors=%d/%d   fallback_rate=%.3f   search_fail_rate=%.3f   decisions=%d"
          % (mirror_errors, n_games, report["mirror_fallback_rate"],
             report["mirror_search_fail_rate"], report["mirror_decisions"]))
    if report["mirror_top_errors"]:
        print("  >>> the policy is FALLING BACK on these exceptions -- fix on the live SDK:")
        for msg, cnt in report["mirror_top_errors"]:
            print("      [%5d x] %s" % (cnt, msg))
    print("vs random  : W/D/L=%d/%d/%d   winrate=%.3f   fallback_rate=%.3f   search_fail_rate=%.3f"
          % (w, d, l, winrate, report["vs_random_fallback_rate"], report["vs_random_search_fail_rate"]))
    print("=" * 64)

    fallback_rate = max(report["mirror_fallback_rate"], report["vs_random_fallback_rate"])
    search_fail_rate = max(report["mirror_search_fail_rate"], report["vs_random_search_fail_rate"])
    search_ok = (not getattr(mod, "USE_SEARCH", False)) or (search_fail_rate <= SEARCH_FAIL_GATE)
    gate_pass = (
        (mirror_errors == 0)
        and (winrate >= WINRATE_GATE)
        and (fallback_rate <= FALLBACK_GATE)
        and search_ok
    )
    if gate_pass:
        print("GATE PASS  PASS  safe-to-submit floor met.")
    else:
        print("GATE FAIL  FAIL  do NOT submit yet:")
        if mirror_errors != 0:
            print("  - mirror produced engine errors (expected 0).")
        if winrate < WINRATE_GATE:
            print("  - winrate %.3f < gate %.2f." % (winrate, WINRATE_GATE))
        if fallback_rate > FALLBACK_GATE:
            print("  - fallback_rate %.3f > gate %.2f: the policy is silently broken on the live SDK"
                  % (fallback_rate, FALLBACK_GATE))
            print("    (0 errors here means the try/except swallowed the bug -- see the exception list above).")
        if not search_ok:
            print("  - search_fail_rate %.3f > gate %.2f while search is enabled."
                  % (search_fail_rate, SEARCH_FAIL_GATE))
            for msg, cnt in (report["mirror_top_search_errors"] + report["vs_random_top_search_errors"])[:6]:
                print("      search [%5d x] %s" % (cnt, msg))
    report["gate_pass"] = gate_pass
    if enforce:
        assert gate_pass, "Phase-1 validation gate failed (see breakdown above)."
    return report


# Raise VALIDATION_GAMES toward a few hundred for a confident pre-submission check.
VALIDATION_GAMES = N_SMOKE_GAMES
GATE_REPORT = run_validation_gate(n_games=VALIDATION_GAMES, enforce=ENFORCE_GATE)

PHASE-1 VALIDATION GATE   (n_games per matchup = 60)
----------------------------------------------------------------
Mirror     : errors=0/60   fallback_rate=0.000   search_fail_rate=0.000   decisions=6984
vs random  : W/D/L=60/0/0   winrate=1.000   fallback_rate=0.000   search_fail_rate=0.000
GATE PASS  PASS  safe-to-submit floor met.


In [19]:
# ============================================================================
# Search A/B  +  attack/branch firing audit
#   (1) Does search actually beat the greedy policy head-to-head? That is the
#       only thing that moves you off the ~600 mirror plateau.
#   (2) Silent dead-ends: a wrong MEGA_BRAVE attackId (983) throws NO exception,
#       it just makes the 270-dmg attack never fire. We print the REAL attackIds
#       seen on a Mega Lucario active so you can fix the constant.
# ============================================================================
def run_search_ab_and_audit(ab_games=20, audit_games=8):
    if CG_DIR is None:
        print("skip: cg/ unavailable; run on Kaggle with cg-lib attached.")
        return None
    try:
        from cg.game import battle_start, battle_select, battle_finish
    except Exception as e:
        print("skip: cg import failed:", repr(e))
        return None

    deck = load_deck_ids()
    mod = import_generated_module()

    # ---- (1) Search vs greedy, same deck, alternating seat ----
    if not getattr(mod, "_SEARCH_AVAILABLE", False):
        print("NOTE: cg search API not importable in this build -> search disabled, A/B skipped.")
    else:
        def toggle_agent(use_search):
            def a(obs_dict):
                mod.USE_SEARCH = use_search
                return mod.agent(obs_dict)
            return a
        search_a, greedy_a = toggle_agent(True), toggle_agent(False)
        mod.diag_reset()
        w = d = l = 0
        for g in range(ab_games):
            if g % 2 == 0:
                res, _, err = play_game(battle_start, battle_select, battle_finish, deck, deck, search_a, greedy_a)
                seat = 0
            else:
                res, _, err = play_game(battle_start, battle_select, battle_finish, deck, deck, greedy_a, search_a)
                seat = 1
            if err is not None or res is None:
                continue
            if res == 2:
                d += 1
            elif res == seat:
                w += 1
            else:
                l += 1
        played = w + d + l
        wr = w / played if played else 0.0
        snap = mod.diag_snapshot()
        print("=" * 64)
        print("SEARCH A/B   (search agent vs greedy agent, identical deck)")
        print("  search W/D/L = %d/%d/%d   winrate = %.3f   (n=%d)" % (w, d, l, wr, played))
        print("  search_used=%d  search_failed=%d  search_fail_rate=%.3f"
              % (snap.get("search_used", 0), snap.get("search_failed", 0), snap.get("search_fail_rate", 0.0)))
        fail_rate = snap.get("search_fail_rate", 0.0)
        if fail_rate > SEARCH_FAIL_GATE:
            print("  -> search is FAILING on the live engine; keep greedy until this is fixed.")
            for msg, cnt in sorted(snap.get("search_errors", snap.get("errors", {})).items(), key=lambda kv: -kv[1])[:5]:
                print("      [%4d x] %s" % (cnt, msg))
        elif wr > 0.55 and played >= 200:
            print("  -> search HELPS under the stricter gate; it is eligible for a search-enabled variant.")
        else:
            print("  -> search is not yet proven; keep the default greedy profile.")
        print("=" * 64)
        mod.USE_SEARCH = False

    # ---- (2) Attack / branch firing audit (greedy mirror) ----
    mod.USE_SEARCH = False
    mod.diag_reset()
    for _ in range(audit_games):
        play_game(battle_start, battle_select, battle_finish, deck, deck, mod.agent, mod.agent)
    snap = mod.diag_snapshot()
    print("ATTACK / BRANCH FIRING AUDIT   (greedy mirror, %d games)" % audit_games)
    print("-" * 64)
    print("MEGA_BRAVE constant in agent =", getattr(mod, "MEGA_BRAVE", "?"),
          "(should match a real Mega Lucario attackId below)")
    for active_id, aids in sorted(snap.get("attack_opts_by_active", {}).items()):
        nm = "Mega Lucario ex(678)" if active_id == 678 else str(active_id)
        print("  active %-22s legal attackIds seen: %s" % (nm, aids))
    print("attackIds actually CHOSEN :", snap.get("attack_ids_chosen", {}))
    print("MEGA_BRAVE seen as option :", snap.get("mega_brave_present", 0), "times")
    print("chosen option-type counts :", snap.get("chosen_types", {}))
    if snap.get("mega_brave_present", 0) == 0:
        print("  !! MEGA_BRAVE was NEVER a legal attack -> 983 is almost certainly wrong.")
        print("     Replace MEGA_BRAVE with the real Mega Lucario(678) attackId listed above.")
    print("=" * 64)
    return snap


AUDIT = run_search_ab_and_audit(ab_games=20, audit_games=8)

SEARCH A/B   (search agent vs greedy agent, identical deck)
  search W/D/L = 7/0/13   winrate = 0.350   (n=20)
  search_used=0  search_failed=8  search_fail_rate=1.000
  -> search is FAILING on the live engine; keep greedy until this is fixed.
      [   8 x] TypeError: search_begin() missing 6 required positional arguments: 'your_deck', 'your_prize', 'opponent_deck', 'opponent_prize', 'opponent_hand', and 'opponent_active'
ATTACK / BRANCH FIRING AUDIT   (greedy mirror, 8 games)
----------------------------------------------------------------
MEGA_BRAVE constant in agent = 983 (should match a real Mega Lucario attackId below)
  active 673                    legal attackIds seen: [976]
  active 674                    legal attackIds seen: [978]
  active 676                    legal attackIds seen: [980]
  active 677                    legal attackIds seen: [981]
  active Mega Lucario ex(678)   legal attackIds seen: [982, 983]
attackIds actually CHOSEN : {982: 40, 980: 15, 983: 18, 978: 3

In [20]:
# ============================================================================
# Crustle wall matchup probe.
# The policy should route through non-ex attackers instead of sending Mega
# Lucario ex into Crustle's ex-damage immunity. The probe checks both the active
# deck and the Crustle-retuned branch against a small wall-deck suite.
# ============================================================================
def _resolve_grass_energy_id(table):
    grass = 1
    if grass in table:
        return grass
    for cid, card in table.items():
        if getattr(card, "name", "").strip().lower() == "basic {g} energy":
            return cid
    return None


def build_crustle_wall_decks():
    if CG_DIR is None:
        return None
    try:
        from cg.api import all_card_data
    except Exception:
        return None
    table = {c.cardId: c for c in all_card_data()}
    grass = _resolve_grass_energy_id(table)
    fixed = [344, 345, 1086, 1147, 1212, 1224, 1264, 1159, 18, 11, 14]
    if grass is None or grass not in table or any(cid not in table for cid in fixed):
        return None

    templates = {
        "public950_wall": {
            344: 4, 345: 4, 1086: 4, 1147: 4, 1212: 4, 1224: 4,
            1264: 4, 1159: 1, 18: 4, 11: 4, 14: 4, grass: 19,
        },
        "energy_dense_wall": {
            344: 4, 345: 4, 1086: 4, 1147: 3, 1212: 4, 1224: 4,
            1264: 3, 1159: 1, 18: 4, 11: 4, 14: 4, grass: 21,
        },
        "trainer_dense_wall": {
            344: 4, 345: 4, 1086: 4, 1147: 4, 1212: 3, 1224: 4,
            1264: 4, 1159: 1, 18: 4, 11: 4, 14: 4, grass: 20,
        },
    }
    decks = {}
    for name, counts in templates.items():
        deck = []
        for cid, n in counts.items():
            deck.extend([cid] * n)
        if len(deck) == 60:
            decks[name] = deck
    return decks or None


def build_crustle_wall_deck():
    decks = build_crustle_wall_decks()
    if not decks:
        return None
    return decks.get("public950_wall") or next(iter(decks.values()))


def crustle_wall_agent_factory(deck):
    def crustle_wall_agent(obs_dict):
        from cg.api import OptionType, SelectContext, to_observation_class
        obs = to_observation_class(obs_dict)
        if obs.select is None:
            return deck
        n = len(obs.select.option)
        minc = max(0, min(obs.select.minCount, n))
        maxc = max(minc, min(obs.select.maxCount, n))
        if n == 0 or maxc == 0:
            return []
        scores = []
        for i, opt in enumerate(obs.select.option):
            score = 0
            if obs.select.context == SelectContext.MAIN:
                priority = {
                    OptionType.ATTACH: 1000,
                    OptionType.EVOLVE: 850,
                    OptionType.PLAY: 650,
                    OptionType.ABILITY: 450,
                    OptionType.ATTACK: 180,
                    OptionType.RETREAT: -20,
                }
                score = priority.get(opt.type, 0)
            elif opt.type == OptionType.CARD:
                score = 1
            scores.append((score, -i, i))
        order = [i for _, _, i in sorted(scores, reverse=True)]
        return order[:maxc] if maxc else order[:minc]
    return crustle_wall_agent


def _probe_candidate_decks():
    decks = {"active": load_deck_ids()}
    try:
        variant = list(DECK_VARIANTS.get("crustle_retuned", []))
        if len(variant) == 60 and variant != decks["active"]:
            decks["crustle_retuned"] = variant
    except Exception:
        pass
    return decks


def run_crustle_wall_probe(n_games=40):
    if CG_DIR is None:
        print("skip Crustle probe: cg/ unavailable in this runtime.")
        return None
    try:
        from cg.game import battle_start, battle_select, battle_finish
    except Exception as e:
        print("skip Crustle probe: cg import failed:", repr(e))
        return None
    wall_decks = build_crustle_wall_decks()
    if not wall_decks:
        print("skip Crustle probe: Crustle deck cards unavailable in this runtime.")
        return None

    candidate_decks = _probe_candidate_decks()
    mod = import_generated_module()
    original_flag = getattr(mod, "CRUSTLE_AWARE", True)
    games_per_pair = max(8, n_games // max(1, len(wall_decks) * len(candidate_decks)))

    crustle_diag_keys = [
        "crustle_seen",
        "crustle_active_seen",
        "crustle_wall_policy_seen",
        "crustle_ex_attack_options",
        "crustle_ex_attack_suppressed",
        "crustle_ex_attack_chosen",
        "crustle_non_ex_attack_chosen",
        "crustle_final_plan_ex_into_wall",
        "crustle_final_plan_non_ex_into_wall",
    ]

    def duel(candidate_deck, opponent_deck, flag):
        if hasattr(mod, "CRUSTLE_AWARE"):
            mod.CRUSTLE_AWARE = flag
        mod.diag_reset()
        opponent = crustle_wall_agent_factory(opponent_deck)
        w = d = l = e = 0
        for g in range(games_per_pair):
            if g % 2 == 0:
                res, _, err = play_game(battle_start, battle_select, battle_finish, candidate_deck, opponent_deck, mod.agent, opponent)
                seat = 0
            else:
                res, _, err = play_game(battle_start, battle_select, battle_finish, opponent_deck, candidate_deck, opponent, mod.agent)
                seat = 1
            if err is not None or res is None:
                e += 1
            elif res == 2:
                d += 1
            elif res == seat:
                w += 1
            else:
                l += 1
        snap = mod.diag_snapshot()
        completed = max(1, w + d + l)
        out = {
            "wins": w,
            "draws": d,
            "losses": l,
            "errors": e,
            "winrate": round((w + 0.5 * d) / completed, 3),
            "fallback_rate": round(snap.get("fallback_rate", 0.0), 4),
        }
        out.update({key: snap.get(key, 0) for key in crustle_diag_keys})
        return out

    report = {}
    print("CRUSTLE WALL PROBE")
    print("  games per pair:", games_per_pair)
    aggregate = {}
    for wall_name, wall_deck in wall_decks.items():
        report[wall_name] = {}
        print("  opponent:", wall_name)
        for cand_name, cand_deck in candidate_decks.items():
            off = duel(cand_deck, wall_deck, False)
            on = duel(cand_deck, wall_deck, True)
            delta = round(on["winrate"] - off["winrate"], 3)
            status = "ok" if on["winrate"] >= 0.5 and delta >= 0 else "check"
            report[wall_name][cand_name] = {
                "aware_off": off,
                "aware_on": on,
                "aware_delta": delta,
                "status": status,
            }
            aggregate.setdefault(cand_name, []).append(on["winrate"])
            print("    %-16s OFF %s" % (cand_name, off))
            print("    %-16s ON  %s  delta=%+.3f  %s" % (cand_name, on, delta, status))
    for cand_name, values in aggregate.items():
        avg_on = round(sum(values) / max(1, len(values)), 3)
        print("  aggregate %-16s aware ON avg_winrate=%.3f" % (cand_name, avg_on))
    if hasattr(mod, "CRUSTLE_AWARE"):
        mod.CRUSTLE_AWARE = original_flag
    return report


CRUSTLE_PROBE = run_crustle_wall_probe(n_games=N_MATCHUP_GAMES)

CRUSTLE WALL PROBE
  games per pair: 30
  opponent: public950_wall
    active           OFF {'wins': 10, 'draws': 0, 'losses': 20, 'errors': 0, 'winrate': 0.333, 'fallback_rate': 0.0, 'crustle_seen': 1653, 'crustle_active_seen': 1222, 'crustle_wall_policy_seen': 0, 'crustle_ex_attack_options': 599, 'crustle_ex_attack_suppressed': 0, 'crustle_ex_attack_chosen': 137, 'crustle_non_ex_attack_chosen': 16, 'crustle_final_plan_ex_into_wall': 0, 'crustle_final_plan_non_ex_into_wall': 0}
    active           ON  {'wins': 14, 'draws': 0, 'losses': 16, 'errors': 0, 'winrate': 0.467, 'fallback_rate': 0.0, 'crustle_seen': 1693, 'crustle_active_seen': 1178, 'crustle_wall_policy_seen': 1693, 'crustle_ex_attack_options': 102, 'crustle_ex_attack_suppressed': 86, 'crustle_ex_attack_chosen': 0, 'crustle_non_ex_attack_chosen': 85, 'crustle_final_plan_ex_into_wall': 0, 'crustle_final_plan_non_ex_into_wall': 301}  delta=+0.134  check
    crustle_retuned  OFF {'wins': 8, 'draws': 0, 'losses': 22, 'errors': 0

In [21]:
if WRITE_SUBMISSION:
    required = [BUILD_DIR / "main.py", BUILD_DIR / "deck.csv"]
    for p in required:
        assert p.exists(), p

    cg_dir = BUILD_DIR / "cg"
    if REQUIRE_CG_IN_ARCHIVE and not cg_dir.exists():
        raise FileNotFoundError(
            "cg/ was not copied. Attach the competition cg-lib input, rerun the cg copy cell, then create submission.tar.gz."
        )

    with tarfile.open(SUBMISSION_PATH, "w:gz") as tar:
        tar.add(BUILD_DIR / "main.py", arcname="main.py")
        tar.add(BUILD_DIR / "deck.csv", arcname="deck.csv")
        if cg_dir.exists():
            for p in sorted(cg_dir.rglob("*")):
                if p.is_file() and "__pycache__" not in p.parts and not p.name.endswith((".pyc", ".pyo")):
                    tar.add(p, arcname=str(p.relative_to(BUILD_DIR)))

    print("created", SUBMISSION_PATH)
    archive_names = validate_submission_bundle(SUBMISSION_PATH, require_cg=REQUIRE_CG_IN_ARCHIVE)
else:
    print("WRITE_SUBMISSION=False; archive not created.")




created /kaggle/working/submission.tar.gz
submission contract OK
archive files: 9
 - cg/__init__.py
 - cg/api.py
 - cg/cg.dll
 - cg/game.py
 - cg/libcg.so
 - cg/sim.py
 - cg/utils.py
 - deck.csv
 - main.py


## Strategic Follow-Up

Next experiments:

1. Run a high-score matchup matrix with alternating seats.
2. Compare active keep-Hero, baseline, and Crustle-retuned decks.
3. A/B target scoring, win-now suppression, and Hero Cape routing.
4. Rebuild search only after the live SDK signature is fully confirmed.
5. Move toward belief-guided search after rule-policy diagnostics are stable.